# Pocket Agent
> **Build a fully functional local coding agent from scratch.**
> 15 chapters · one notebook · no API key needed.

This notebook runs on **Google Colab** by default using the free Gemini model — no account, no API key, no local install.
It also works with a **local Ollama** installation if you prefer to run everything on your own machine.
The only cells that differ between the two are the setup cells in Chapter 1. Everything else is identical.

Each chapter adds exactly one capability.
Run a chapter's cells top-to-bottom and you get a working agent at the end of that chapter.

| Ch | Title | What you gain |
|----|-------|---------------|
| 1  | Setup | Configure the notebook for Colab or Ollama |
| 2  | Hello LLM | Talk to an LLM from Python |
| 3  | Context is a Budget | Understand why context management exists |
| 4  | Give it a Map | Load AGENTS.md as an advisory project manifest |
| 5  | Glob + JIT Reads | Navigate a file tree without bulk-loading files |
| 6  | Fuzzy Scoring | Rank retrieved files, not just find them |
| 7  | Grep | Find code by content, not just filename |
| 8  | Microcompaction | Hot/cold storage — survive long sessions |
| 9  | Semantic RAG | Retrieval that understands meaning, not just keywords |
| 10 | Full Pipeline | One `run(query, repo)` call does everything |
| 11 | Write + Diff | The agent modifies files on disk |
| 12 | Agent Loop | Autonomous read → plan → write → verify loop |
| 13 | Test Generation | Agent writes and verifies its own tests |
| 14 | Adding a Capability | The three-step pattern for extending the agent |
| 15 | Reflection | ReAct loop — observe each step, revise the plan |

## Chapter 1 — Setup

## Cleanup

Run the cell below if you want to reset to a clean slate before re-running the notebook from the top.

**You only need this if you've run the notebook before.** Here's why: when the agent is asked to write a file that already exists with correct content, `apply_patch` compares the current file against what it would write — and if they match, it produces no diff and makes no changes. The agent does the right thing, but nothing visible happens, which can be confusing on a second run.

Deleting the generated files forces the agent to create them fresh, so you see the full write → diff → verify cycle as intended.

In [ ]:
# clean up
from pathlib import Path
import shutil

def cleanup():
    for path in ["docs", "sample_project"]:
        shutil.rmtree(Path(path), ignore_errors=True)

cleanup()

## Ollama

### 1.1 Install Ollama

Download and install Ollama for your platform from **[ollama.com/download](https://ollama.com/download)**.

Once installed, start the Ollama daemon if it isn't already running:

```bash
ollama serve
```

> On macOS and Windows, Ollama starts automatically when you open the app.  
> On Linux, run `ollama serve` in a separate terminal and leave it open.

Browse available models at **[ollama.com/library](https://ollama.com/library)**.  
Cloud models (tagged `cloud`) run on Ollama's servers — no GPU or download required, just `ollama signin` first.

### 1.2 Pull a model

Set `MODEL_TO_PULL` in the cell below to whichever model you want, then run it.  
It calls `ollama pull` for you — no terminal needed.

**Recommendations:**

| Model | Size |
|-------|------|
| `qwen3-coder-next:cloud` | cloud (no download) |
| `qwen3-coder-next` | ~52 GB (local GPU) |

The cell also pulls `nomic-embed-text` for Chapter 9 semantic search — it's small (~274 MB) and converts text into meaning-vectors rather than generating language.

In [ ]:
import subprocess, sys

# ── Ollama only — skip this cell if you are on Google Colab ──────────────────
# On Colab, the model is provided by Google Colab AI and nothing needs pulling.
# On a local machine with Ollama, set MODEL_TO_PULL to whichever model you want
# and run this cell to download it.

MODEL_TO_PULL = "qwen3-coder-next:cloud"   # ← edit me (Ollama only)

def _pull(model: str) -> None:
    """Pull an Ollama model, suppressing the spinner output that clutters notebooks."""
    print(f"Pulling {model} ...", end=" ", flush=True)
    result = subprocess.run(
        ["ollama", "pull", model],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    if result.returncode == 0:
        print("done ✓")
    else:
        print("FAILED ✗")
        raise RuntimeError(f"ollama pull {model} exited with code {result.returncode}")

try:
    _pull(MODEL_TO_PULL)
    _pull("nomic-embed-text")   # needed for Chapter 9 semantic search
    print("\nAll models ready. Proceed to the next cell.")
except FileNotFoundError:
    # ollama command not found — running on Colab, nothing to pull
    print("Ollama not found — using Colab backend. This cell is a no-op.")

### 1.3 Ollama configuration — local users only, skip on Colab

In [ ]:
import os
from types import SimpleNamespace as _NS

BACKEND       = "ollama"
EMBED_BACKEND = "ollama"
REPO_ROOT     = "."
REBUILD_INDEX = False

_ollama = _NS(
    base_url = "http://localhost:11434",
    model    = globals().get("MODEL_TO_PULL", ""),   # set in cell 1.2
    embed    = "nomic-embed-text",
    context_windows = {
        "qwen3-coder-next": 262144,
        "qwen3-coder":      131072,
        "qwen4.5":           32768,
        "qwen3":             32768,
        "llama4.2":          32768,
        "llama4.1":          32768,
        "mistral":           32768,
        "codellama":          4096,
        "gemma3":            32768,
        "devstral-small-2":  32768,
    },
)

_base        = _ollama.model.split(":")[0]
if _ollama.model and _base not in _ollama.context_windows:
    print(f"WARNING: '{_base}' not in context_windows — defaulting to 32,768 tokens.")
    print(f"  Run: ollama show {_ollama.model}  and look for 'context length'")
TOKEN_BUDGET = _ollama.context_windows.get(_base, 32768)
SAFE_NUM_CTX = int(os.getenv("OLLAMA_NUM_CTX", str(TOKEN_BUDGET)))

print(f"Backend      : Ollama ({_ollama.model or 'not set'})")
print(f"Embeddings   : {_ollama.embed}")
print(f"Token budget : {TOKEN_BUDGET:,}")
print(f"Repo root    : {REPO_ROOT}")

In [ ]:
import requests   # HTTP client — used throughout the notebook

def ping_ollama() -> tuple[bool, list[str] | str]:
    """
    Check that the Ollama server is reachable and the chosen model is pulled.

    Returns (ok, info) where ok=True means Ollama is up and info is a list
    of pulled model names; ok=False means something went wrong and info is
    the error message.

    Only meaningful when BACKEND == "ollama" — skipped automatically on Colab.
    """
    try:
        r = requests.get(f"{_ollama.base_url}/api/tags", timeout=5)
        r.raise_for_status()
        models = [m["name"] for m in r.json().get("models", [])]
        return True, models
    except Exception as exc:
        return False, str(exc)


In [ ]:
if BACKEND == "ollama":
    ok, info = ping_ollama()
    if ok:
        print(f"Ollama is running.")
        print(f"Pulled models: {info}")
        if _ollama.model and _ollama.model not in " ".join(info):
            print(f"\n  WARNING: '{_ollama.model}' not found.")
            print(f"  Run: ollama pull {_ollama.model}")
    else:
        print(f"Cannot reach Ollama at {_ollama.base_url}")
        print(f"Error: {info}")
        print("Start it with: ollama serve")
else:
    print("Google Colab — Ollama check skipped.")

## Google Colab

In [ ]:
import os
from types import SimpleNamespace as _NS

BACKEND       = "colab"
EMBED_BACKEND = "sentence-transformers"
REPO_ROOT     = "."
REBUILD_INDEX = False

COLAB_MODEL   = "google/gemini-2.5-flash-lite"   # free tier
# COLAB_MODEL = "google/gemini-2.5-flash"         # ← uncomment for paid tier

TOKEN_BUDGET  = 100_000   # Gemini supports ~1M; conservative for our budget logic
SAFE_NUM_CTX  = TOKEN_BUDGET

# Empty stub — keeps helper functions that reference _ollama.* from crashing
# if they're ever called outside their intended backend branch.
_ollama = _NS(base_url="", model="", embed="", context_windows={})

print(f"Backend      : Google Colab AI ({COLAB_MODEL})")
print(f"Embeddings   : sentence-transformers / all-MiniLM-L6-v2")
print(f"Token budget : {TOKEN_BUDGET:,}")
print(f"Repo root    : {REPO_ROOT}")

---
## Chapter 2 — Hello LLM

**Goal:** create the two things everything else depends on — a `chat()` function that talks to the LLM, and a status panel that shows what happened after each call.

| Section | What it builds |
|---------|----------------|
| 2.1 | Install dependencies |
| 2.2 | `chat()` — the single function that calls the LLM |
| 2.3 | Status panel — a reusable display for every chapter |
| 2.4 | Try it |

### 2.1 Dependencies

Chapter 2 only needs `requests` (HTTP).
`IPython.display` is part of Jupyter itself — no install needed.
Later chapters will `pip install` their own additions at the top of their section.


In [ ]:
# Ensure Chapter 2 dependencies are installed in the current Python environment.
import subprocess, sys

_CH2_DEPS = ["requests"]   # requests: HTTP calls to Ollama
                            # IPython.display is bundled with Jupyter — no install needed

for _pkg in _CH2_DEPS:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", _pkg],
        stdout=subprocess.DEVNULL,
    )

print("Chapter 2 dependencies ready:", _CH2_DEPS)


### 2.2 `chat()` — the only function that calls the LLM

`chat()` is the single gateway to the model. Every other function in the notebook calls this one — nothing else talks to the LLM directly. It takes a list of messages and returns the reply text and a token count.

On Ollama, it posts to the local `/api/chat` endpoint. On Google Colab, it calls `google.colab.ai.generate_text()`. The rest of the notebook never needs to know which one it is.

In [ ]:
def _messages_to_prompt(messages: list[dict]) -> str:
    """
    Flatten a messages list into a single labelled string.

    google-colab-ai accepts a plain string rather than a structured messages
    list, so we convert before sending.  The role labels ([System], [User],
    [Assistant]) tell Gemini how to interpret each turn.
    """
    parts = []
    for m in messages:
        role    = m["role"].capitalize()   # System / User / Assistant
        content = m["content"]
        parts.append(f"[{role}]\n{content}")
    return "\n\n".join(parts)


def _chat_ollama(messages: list[dict], model: str) -> tuple[str, int]:
    """Send messages to a local Ollama server; auto-continue if cut off mid-reply."""
    full_reply   = ""
    total_tokens = 0
    msgs         = list(messages)
    while True:
        payload = {
            "model":    model,
            "messages": msgs,
            "stream":   False,
            "options":  {"num_ctx": min(TOKEN_BUDGET, SAFE_NUM_CTX)},
        }
        r = requests.post(f"{_ollama.base_url}/api/chat", json=payload, timeout=180)
        r.raise_for_status()
        data         = r.json()
        chunk        = data["message"]["content"]
        full_reply  += chunk
        total_tokens += data.get("prompt_eval_count", 0) + data.get("eval_count", 0)
        if data.get("done_reason", "stop") != "length":
            break
        msgs = msgs + [
            {"role": "assistant", "content": chunk},
            {"role": "user",      "content": "Continue."},
        ]
    return full_reply, total_tokens


def _chat_colab(messages: list[dict], model: str) -> tuple[str, int]:
    """
    Send messages to Google Colab AI (no API key required).

    Gemini's context window is ~1M tokens, so the length-continuation loop
    in chat() will never fire in practice.
    """
    from google.colab import ai
    prompt = _messages_to_prompt(messages)
    reply  = ai.generate_text(prompt, model_name=model)
    # Colab AI doesn't return token counts.  1 token ≈ 4 characters is a
    # reasonable estimate — count_tokens() in Chapter 3 does the same thing.
    tokens = (len(prompt) + len(reply)) // 4
    return reply, tokens


def chat(
    messages: list[dict],
    model:    str | None = None,
) -> tuple[str, int]:
    """
    Send *messages* to the configured LLM, return (reply_text, tokens_used).

    Dispatches to the Ollama or Colab backend based on the global BACKEND
    variable set in the config cell.
    """
    if BACKEND == "colab":
        return _chat_colab(messages, model or COLAB_MODEL)
    return _chat_ollama(messages, model or _ollama.model)

### 2.3 The status panel

The panel is the same in every chapter — only the data passed to it changes. It shows token usage, the files loaded, and the model's reply side-by-side so you can see cost and output together.

In [ ]:
from IPython.display import HTML, Markdown, display


def show_rule(title: str = "") -> None:
    """Render a horizontal rule with a centred title — native notebook HTML."""
    safe = title.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML(
        f'<div style="display:flex;align-items:center;margin:14px 0 6px;gap:8px;font-family:monospace">'
        f'<hr style="flex:1;border:none;border-top:1px solid #d0d7de;margin:0">'
        f'<b style="color:#57606a;white-space:nowrap;font-size:0.9rem">{safe}</b>'
        f'<hr style="flex:1;border:none;border-top:1px solid #d0d7de;margin:0">'
        f'</div>'
    ))


def show_details(summary: str, body: str, open: bool = False) -> None:
    """
    Render a collapsible <details> block in Colab / JupyterLab.

    The summary line is always visible; clicking it expands the full body.
    Pass open=True to start expanded (useful for error output that needs
    immediate attention).

    Parameters
    ----------
    summary : short label shown on the collapsed row, e.g. "✅ BASH pytest"
    body    : the full text shown when expanded
    open    : if True the block starts in the expanded state
    """
    open_attr  = " open" if open else ""
    safe_label = summary.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    safe_body  = body.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML(
        f'<details{open_attr} style="margin:3px 0">'
        f'<summary style="font-size:0.82rem;color:#57606a;cursor:pointer;'
        f'padding:2px 6px;border-radius:3px;list-style:disclosure-closed">'
        f'{safe_label}</summary>'
        f'<pre style="font-size:0.78rem;color:#333;background:#f6f8fa;'
        f'padding:8px 12px;border-radius:4px;overflow:auto;'
        f'white-space:pre-wrap;margin:4px 0 6px;border:1px solid #d0d7de">'
        f'{safe_body}</pre></details>'
    ))


def show_panel(
    query:        str,
    token_used:   int,
    token_budget: int | None    = None,   # None → read TOKEN_BUDGET at call time
    files:        list[dict] | None = None,
    strategy:     str           = "none",
    prompt_size:  int           = 0,
) -> None:
    """
    Render the Pocket Agent status panel as HTML in the notebook output.

    Parameters
    ----------
    query        : the user's query (shown in the panel title)
    token_used   : tokens consumed so far
    token_budget : total context-window size (defaults to the global TOKEN_BUDGET)
    files        : list of dicts {"path": str, "hot": bool}
    strategy     : retrieval strategy name
    prompt_size  : estimated assembled-prompt size in tokens
    """
    # Look up the global TOKEN_BUDGET at call time so temporary overrides
    # (e.g. demo cells that shrink TOKEN_BUDGET) are reflected correctly.
    budget = token_budget if token_budget is not None else TOKEN_BUDGET

    files = files or []
    pct   = token_used / budget if budget > 0 else 0
    bar_w = min(int(pct * 200), 200)
    bar_color = (
        "#2da44e" if pct < 0.70 else
        "#bf8700" if pct < 0.85 else
        "#cf222e"
    )

    file_rows = ""
    for f in files:
        if f["hot"]:
            badge = '<span style="background:#cf222e;color:#fff;border-radius:3px;padding:1px 6px;font-size:0.78rem;font-weight:bold">HOT</span>'
        else:
            badge = '<span style="background:#0969da;color:#fff;border-radius:3px;padding:1px 6px;font-size:0.78rem;font-weight:bold">COLD</span>'
        path  = f["path"].replace("&","&amp;").replace("<","&lt;")
        file_rows += f'<div style="margin:2px 0">{badge}&nbsp;&nbsp;<code style="font-size:0.85rem">{path}</code></div>'

    if not file_rows:
        file_rows = '<span style="color:#8c959f;font-style:italic">none</span>'

    short_q = query[:80] + ("…" if len(query) > 80 else "")
    short_q = short_q.replace("&","&amp;").replace("<","&lt;")

    html = f"""
<div style="border:1px solid #d0d7de;border-radius:8px;padding:14px 18px;
            margin:8px 0;background:#f6f8fa;font-family:monospace;font-size:0.88rem">
  <div style="font-weight:bold;color:#0969da;margin-bottom:10px">
    Pocket Agent&nbsp;&nbsp;·&nbsp;&nbsp;<em style="font-weight:normal;color:#57606a">{short_q}</em>
  </div>
  <table style="border-collapse:collapse;width:100%">
    <tr>
      <td style="padding:4px 14px 4px 0;color:#57606a;vertical-align:middle;white-space:nowrap">Token Budget</td>
      <td style="vertical-align:middle">
        <div style="display:inline-flex;align-items:center;gap:10px">
          <div style="width:200px;height:10px;border-radius:5px;background:#e0e0e0;overflow:hidden">
            <div style="width:{bar_w}px;height:100%;background:{bar_color};transition:width 0.3s"></div>
          </div>
          <span style="color:#57606a;font-size:0.82rem">{token_used:,}&nbsp;/&nbsp;{budget:,} tokens</span>
        </div>
      </td>
    </tr>
    <tr>
      <td style="padding:4px 14px 4px 0;color:#57606a;vertical-align:top">Retrieved</td>
      <td style="vertical-align:top">{file_rows}</td>
    </tr>
    <tr>
      <td style="padding:4px 14px 4px 0;color:#57606a">Strategy</td>
      <td style="color:#8250df;font-weight:bold">{strategy}</td>
    </tr>
    <tr>
      <td style="padding:4px 14px 4px 0;color:#57606a">Prompt&nbsp;size</td>
      <td style="color:#57606a">{prompt_size:,} tokens</td>
    </tr>
  </table>
</div>"""
    display(HTML(html))


def show_reply(text: str) -> None:
    """Render the model's reply as markdown. Falls back to print() outside Jupyter."""
    try:
        display(Markdown(text))
    except Exception:
        print(text)


def show_log(log: list[dict]) -> None:
    """
    Pretty-print the step log returned by agent_loop().

    Each entry in the log is  {**step, "result": dict, "accepted": bool}.
    Use this after running agent_loop() to inspect what the agent actually did:

        log = agent_loop(task="...", repo_root="sample_project")
        show_log(log)
    """
    if not log:
        print("(empty log)")
        return
    for i, entry in enumerate(log, 1):
        action = entry.get("action", "?").upper()
        desc   = entry.get("step",   "")
        result = entry.get("result", entry.get("status", {}))

        if isinstance(result, dict):
            ok     = result.get("ok", True)
            output = str(result.get("output", result.get("diff", "")))
            icon   = "✅" if ok else "❌"
            label  = f"{icon} Step {i}/{len(log)}  {action}  {desc}"
            show_details(label, output[:3000] + ("…" if len(output) > 3000 else ""), open=not ok)
        else:
            show_details(f"Step {i}/{len(log)}  {action}  {desc}", str(result)[:3000])

### 2.4 Try it

Send a single question to the LLM and observe the panel before and after. The pre-call panel shows an estimate of what the query will cost; the post-call panel shows the actual tokens used.

In [ ]:
query    = "What is a context window, and why does its size matter?"
messages = [{"role": "user", "content": query}]

# Rough pre-call estimate: 1 token ≈ 4 characters
prompt_size = len(query) // 4

show_rule("Before LLM call")
show_panel(
    query        = query,
    token_used   = prompt_size,
    strategy     = "none",
    prompt_size  = prompt_size,
)

print("\nSending query to LLM…\n")
reply, tokens_used = chat(messages)

show_rule("After LLM call")
show_panel(
    query       = query,
    token_used  = tokens_used,
    strategy    = "none",
    prompt_size = prompt_size,
)

show_reply(reply)


---
## Chapter 3 — Context is a Budget

**Goal:** understand *why* context management exists before we build any of it.

Every token you load — user message, system prompt, file content, conversation history — occupies space in the context window. When that space runs out, one of two things happens: the model silently truncates early content (it forgets the beginning of the conversation), or the API returns an error.

A coding agent that naively loads every file in a repo will hit the wall on the first non-trivial query.

**You will:**
- Replace the inline `len // 4` guess with a named `count_tokens()` function
- Watch a multi-turn conversation burn through its budget turn by turn
- Measure the token cost of real files on disk
- See the panel turn yellow then red as budget pressure rises
- Understand the three strategies agents use to stay inside the window (foreshadowing Ch 3–8)

### 3.1 `count_tokens()` — one place to estimate token cost

If we were using OpenAI models, we could use tiktoken for exact token counts.
For a provider‑agnostic approach, we’d need a tokenizer from the model’s ecosystem (often a Hugging Face tokenizer) — which is heavier to load and configure.

The `÷ 4` heuristic (1 token ≈ 4 characters in English prose) is accurate to
within ~15 % for most code and documentation. That's precise enough to drive
a budget bar — we don't need exact counts, we need *early warning*.

Wrapping it in a named function means we can swap the implementation once
(e.g. call Ollama's `/api/tokenize` endpoint) without touching any of the
chapter code that calls it.

In [ ]:
def count_tokens(text: str) -> int:
    """
    Estimate token count for *text*.

    Uses the 4-characters-per-token heuristic — accurate to ~15 % for
    English prose and source code.  Good enough to drive a budget bar.

    Swap the body for a real tokeniser call if you need precision:
        # example: use Ollama's tokenise endpoint
        # r = requests.post(f"{OLLAMA_BASE_URL}/api/tokenize",
        #                   json={"model": OLLAMA_MODEL, "content": text})
        # return len(r.json()["tokens"])
    """
    return max(1, len(text) // 4)

In [ ]:
# Quick sanity check
_sample = "The quick brown fox jumps over the lazy dog."
print(f"Sample: {len(_sample)} chars → {count_tokens(_sample)} tokens  "
      f"(GPT-4 tokeniser gives ~11 for this sentence)")

### 3.2 Watching a conversation burn its budget

Each turn appends the user message *and* the assistant reply to the
`messages` list before the next call.  Ollama sees the full list every time —
that's how it maintains conversation context, but it also means every reply
you get back is added to your next prompt.

Run this cell and watch the budget bar grow with each exchange.

In [ ]:
TURNS = [
    "What is a Python list comprehension? Give a one-line example.",
    "Now show a dictionary comprehension that squares numbers 1–5.",
    "What is the difference between a shallow copy and a deep copy?",
    "Show me a minimal example where a shallow copy causes a bug.",
]

messages: list[dict] = []
tokens_used = 0

for i, question in enumerate(TURNS, start=1):
    messages.append({"role": "user", "content": question})

    # Estimate prompt size before the call
    prompt_text = " ".join(m["content"] for m in messages)
    prompt_size = count_tokens(prompt_text)

    show_rule(f"Turn {i} — before call")
    show_panel(
        query        = question,
        token_used   = tokens_used,
        strategy     = "multi-turn",
        prompt_size  = prompt_size,
    )

    reply, tokens_used = chat(messages)
    messages.append({"role": "assistant", "content": reply})

    show_rule(f"Turn {i} — after call")
    show_panel(
        query       = question,
        token_used  = tokens_used,
        strategy    = "multi-turn",
        prompt_size = prompt_size,
    )
    print(f"\nReply: {reply[:200]}{'…' if len(reply) > 200 else ''}\n")


### 3.3 Sample project setup

Chapter 3.4 scans real files to show token costs. Run the cells below once to create a small fake Python project under `sample_project/` — these files are used throughout all later chapters.

In [ ]:
from pathlib import Path
Path("sample_project/utils").mkdir(parents=True, exist_ok=True)
Path("sample_project/tests").mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile sample_project/main.py
"""Entry point for the JSON-to-CSV converter tool."""

from utils.parser import parse_json
from utils.formatter import to_csv
from utils.validator import validate_schema
import sys


def main(input_path: str, output_path: str, schema_path: str | None = None) -> None:
    with open(input_path) as f:
        raw = f.read()

    records = parse_json(raw)

    if schema_path:
        with open(schema_path) as f:
            schema = f.read()
        validate_schema(records, schema)

    csv_text = to_csv(records)

    with open(output_path, "w") as f:
        f.write(csv_text)

    print(f"Wrote {len(records)} records to {output_path}")


if __name__ == "__main__":
    main(*sys.argv[1:])

In [ ]:
%%writefile sample_project/utils/parser.py
"""Parse a JSON string into a list of flat dicts."""

import json


def parse_json(raw: str) -> list[dict]:
    """
    Accept a JSON string that is either:
    - a list of objects   → returned as-is
    - a single object     → wrapped in a list
    - a list of scalars   → each scalar wrapped as {"value": scalar}

    Raises ValueError on malformed input.
    """
    try:
        data = json.loads(raw)
    except json.JSONDecodeError as exc:
        raise ValueError(f"Invalid JSON: {exc}") from exc

    if isinstance(data, list):
        return [_flatten(item) if isinstance(item, dict) else {"value": item}
                for item in data]
    if isinstance(data, dict):
        return [_flatten(data)]

    raise ValueError(f"Expected a JSON object or array, got {type(data).__name__}")


def _flatten(obj: dict, prefix: str = "", sep: str = ".") -> dict:
    """Recursively flatten nested dicts: {"a": {"b": 1}} → {"a.b": 1}."""
    result = {}
    for key, value in obj.items():
        full_key = f"{prefix}{sep}{key}" if prefix else key
        if isinstance(value, dict):
            result.update(_flatten(value, full_key, sep))
        else:
            result[full_key] = value
    return result

In [ ]:
%%writefile sample_project/utils/formatter.py
"""Format a list of flat dicts as CSV text."""

import csv
import io


def to_csv(records: list[dict], delimiter: str = ",") -> str:
    """
    Convert *records* (list of flat dicts) to a CSV string.

    All keys across all records are used as headers.
    Missing values are rendered as empty strings.
    """
    if not records:
        return ""

    # Collect all headers preserving first-seen order
    headers: list[str] = []
    seen: set[str] = set()
    for rec in records:
        for key in rec:
            if key not in seen:
                headers.append(key)
                seen.add(key)

    buf = io.StringIO()
    writer = csv.DictWriter(buf, fieldnames=headers,
                            delimiter=delimiter, extrasaction="ignore")
    writer.writeheader()
    for rec in records:
        writer.writerow({h: rec.get(h, "") for h in headers})

    return buf.getvalue()

In [ ]:
%%writefile sample_project/utils/validator.py
"""Validate a list of records against a simple JSON schema."""

import json


def validate_schema(records: list[dict], schema_raw: str) -> None:
    """
    Validate each record against *schema_raw* (a JSON object mapping
    field names to expected types: "string", "number", "boolean").

    Raises TypeError on the first violation found.
    """
    schema: dict[str, str] = json.loads(schema_raw)
    type_map = {"string": str, "number": (int, float), "boolean": bool}

    for i, record in enumerate(records):
        for field, expected_type_name in schema.items():
            if field not in record:
                raise TypeError(
                    f"Record {i}: missing required field '{field}'"
                )
            expected = type_map.get(expected_type_name)
            if expected and not isinstance(record[field], expected):
                raise TypeError(
                    f"Record {i}: field '{field}' expected {expected_type_name}, "
                    f"got {type(record[field]).__name__}"
                )

In [ ]:
%%writefile sample_project/tests/test_parser.py
"""Unit tests for utils/parser.py."""

import pytest
from utils.parser import parse_json, _flatten


class TestParseJson:
    def test_list_of_dicts(self):
        result = parse_json('[{"a": 1}, {"a": 2}]')
        assert result == [{"a": 1}, {"a": 2}]

    def test_single_dict_wrapped(self):
        result = parse_json('{"x": 10}')
        assert result == [{"x": 10}]

    def test_list_of_scalars(self):
        result = parse_json('[1, 2, 3]')
        assert result == [{"value": 1}, {"value": 2}, {"value": 3}]

    def test_invalid_json_raises(self):
        with pytest.raises(ValueError, match="Invalid JSON"):
            parse_json("{not valid}")

    def test_unexpected_type_raises(self):
        with pytest.raises(ValueError, match="Expected"):
            parse_json('"just a string"')


class TestFlatten:
    def test_nested_dict(self):
        assert _flatten({"a": {"b": 1}}) == {"a.b": 1}

    def test_deeply_nested(self):
        assert _flatten({"a": {"b": {"c": 42}}}) == {"a.b.c": 42}

    def test_flat_dict_unchanged(self):
        assert _flatten({"x": 1, "y": 2}) == {"x": 1, "y": 2}

In [ ]:
%%writefile sample_project/tests/test_formatter.py
"""Unit tests for utils/formatter.py."""

from utils.formatter import to_csv


def test_empty_input():
    assert to_csv([]) == ""


def test_single_record():
    result = to_csv([{"name": "Alice", "age": 30}])
    lines = result.strip().splitlines()
    assert lines[0] == "name,age"
    assert lines[1] == "Alice,30"


def test_missing_fields_become_empty():
    records = [{"a": 1, "b": 2}, {"a": 3}]
    result = to_csv(records)
    lines = result.strip().splitlines()
    assert lines[0] == "a,b"
    assert lines[2] == "3,"


def test_header_order_follows_first_seen():
    records = [{"z": 1, "a": 2}]
    result = to_csv(records)
    assert result.startswith("z,a")

### 3.4 What files actually cost

A coding agent loads source files to answer questions about them.
Let's measure what that costs in tokens.

We'll scan the files in `REPO_ROOT`, count their token cost, and show
how quickly a naive "load everything" strategy would exhaust the budget.

In [ ]:
import os
from pathlib import Path

# File extensions we'd want to load for a coding question
CODE_EXTENSIONS = {".py", ".js", ".ts", ".go", ".rs", ".java",
                   ".c", ".cpp", ".h", ".md", ".txt", ".yaml", ".toml", ".json"}

def scan_repo_costs(root: str) -> list[dict]:
    """
    Walk *root* and return a list of dicts:
        {"path": relative_path, "bytes": int, "tokens": int}
    for every source file found, sorted by token cost descending.
    """
    results = []
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [d for d in dirnames
                       if not d.startswith(".") and d not in
                       {"__pycache__", "node_modules", ".git", "venv", ".venv"}]
        for fname in filenames:
            full = Path(dirpath) / fname
            if full.suffix.lower() not in CODE_EXTENSIONS:
                continue
            try:
                text = full.read_text(errors="ignore")
            except OSError:
                continue
            results.append({
                "path":   str(full.relative_to(root)),
                "bytes":  len(text.encode()),
                "tokens": count_tokens(text),
            })
    return sorted(results, key=lambda x: x["tokens"], reverse=True)


In [ ]:
file_costs = scan_repo_costs(REPO_ROOT)

# ── HTML table ──────────────────────────────────────────────────────────────
rows = ""
for f in file_costs[:20]:
    pct = f["tokens"] / TOKEN_BUDGET * 100
    color = "#2da44e" if pct < 10 else ("#bf8700" if pct < 30 else "#cf222e")
    rows += (
        f'<tr><td style="font-family:monospace;padding:3px 12px 3px 0">{f["path"]}</td>'
        f'<td style="text-align:right;padding:3px 8px">{f["bytes"]:,}</td>'
        f'<td style="text-align:right;padding:3px 8px;color:{color};font-weight:bold">{f["tokens"]:,}</td>'
        f'<td style="text-align:right;padding:3px 8px;color:{color}">{pct:.1f}%</td></tr>'
    )

total = sum(f["tokens"] for f in file_costs)
fits  = total <= TOKEN_BUDGET
summary_color = "#2da44e" if fits else "#cf222e"
summary = "YES ✓" if fits else "NO ✗"

display(HTML(f"""
<div style="font-family:monospace;font-size:0.88rem">
  <b>File token costs in '{REPO_ROOT}'</b>
  <table style="border-collapse:collapse;margin-top:8px">
    <tr style="border-bottom:1px solid #d0d7de;color:#57606a">
      <th style="text-align:left;padding:3px 12px 3px 0">File</th>
      <th style="text-align:right;padding:3px 8px">Bytes</th>
      <th style="text-align:right;padding:3px 8px">Tokens</th>
      <th style="text-align:right;padding:3px 8px">% of budget</th>
    </tr>
    {rows}
  </table>
  <div style="margin-top:8px;color:#57606a">
    Total across {len(file_costs)} files:&nbsp;
    <span style="color:{summary_color};font-weight:bold">{total:,} tokens</span>
    &nbsp;·&nbsp; Budget: {TOKEN_BUDGET:,}
    &nbsp;·&nbsp; Fits in one prompt:&nbsp;
    <span style="color:{summary_color};font-weight:bold">{summary}</span>
  </div>
</div>
"""))

### 3.5 Hitting the wall — what over-budget looks like

Let's simulate what happens when a naive agent stuffs too much into one prompt.
We'll build a fake "load everything" prompt, measure it, and show the panel
turning red — without actually sending it to Ollama (no point wasting tokens
on a prompt that would be truncated or error).

In [ ]:
def _simulate_overbudget() -> None:
    """
    Build a hypothetical prompt that exceeds TOKEN_BUDGET and show
    the panel at 50 %, 85 %, and 110 % capacity — no LLM call needed.
    """
    # Each file is a *fraction* of the budget so accumulation crosses 100 %
    # gradually: file 1 → ~50 %, file 2 → ~85 %, file 3 → ~112 %
    fake_files = [
        {"name": "src/parser.py",       "tokens": int(TOKEN_BUDGET * 0.50)},
        {"name": "src/compiler.py",     "tokens": int(TOKEN_BUDGET * 0.25)},
        {"name": "src/runtime.py",      "tokens": int(TOKEN_BUDGET * 0.27)},
        {"name": "tests/test_parser.py","tokens": int(TOKEN_BUDGET * 0.20)},
    ]

    loaded_files = []
    accumulated  = 0
    query        = "Explain how the compiler hands off to the runtime"

    for i, f in enumerate(fake_files):
        accumulated += f["tokens"]
        loaded_files.append({"path": f["name"], "hot": True})

        pct = accumulated / TOKEN_BUDGET
        label = (
            "comfortable" if pct < 0.70 else
            "warning"     if pct < 1.00 else
            "OVER BUDGET"
        )
        show_rule(f"After loading {i+1} file(s) — {label}")
        show_panel(
            query        = query,
            token_used   = accumulated,
            files        = loaded_files,
            strategy     = "naive load-all",
            prompt_size  = accumulated,
        )

        if accumulated > TOKEN_BUDGET:
            print("\n⚠  Prompt exceeds context window.")
            print("Ollama will truncate the earliest messages — the agent may")
            print("forget the system prompt or earlier file contents entirely.\n")
            break

In [ ]:
_simulate_overbudget()

### 3.6 The three strategies — what's coming next

Every technique in Chapters 4–9 is a variation of one of these three responses
to budget pressure:

| Strategy | What it does | Where we build it |
|----------|-------------|-------------------|
| **Be selective** | Only load files relevant to the query | Ch 4 (manifest), Ch 5 (glob + JIT), Ch 6 (fuzzy), Ch 7 (grep) |
| **Be lazy** | Load file *summaries* first, full content only on demand | Ch 5 JIT reads |
| **Evict & compress** | Move cold context to a summary store; keep hot context small | Ch 8 microcompaction, Ch 9 semantic RAG |

The key insight: **a good agent spends tokens like a good programmer spends CPU** — only when necessary, and on the thing most likely to matter.

Chapter 4 introduces the first tool: an `AGENTS.md` manifest that tells the agent which files matter *before* it even looks at the file tree.


---
## Chapter 4 — Give it a Map

**Goal:** before the agent looks at a single file, hand it a curated index of the repo so it already knows what exists and what matters.

That index is `AGENTS.md` — a small markdown file you commit at the repo root.
It costs a fixed, known number of tokens on every call (cheap), and it dramatically reduces the search space for every retrieval strategy we build later.

**You will:**
- Create an `AGENTS.md` for this repo
- Write `load_manifest()` — reads the file, extracts mentioned paths
- Write `ask_with_manifest()` — prepends the manifest to every prompt
- See the panel report the manifest's token cost before any file is loaded

### 4.1 What goes in AGENTS.md?

An `AGENTS.md` is just a markdown file. It has no required schema — the agent
reads it as plain text. The convention is:

- **Entry points** — where execution starts
- **Key modules** — what each important file does in one line
- **Ownership sections** — "questions about X → look in Y"
- **Off-limits** — generated files, build artefacts the agent should skip

Keep it under ~400 tokens. If it grows larger than that, it starts eating the
budget it was meant to protect.

The cell below uses `%%writefile` to create `AGENTS.md` on disk.
`%%writefile path` is a Jupyter magic: instead of running the cell,
it saves the cell body verbatim to the given path. The file will live next to
this notebook in `REPO_ROOT`.

In [ ]:
%%writefile AGENTS.md
# AGENTS.md — Pocket Agent project map

## What this repo is
A Jupyter notebook tutorial that builds a local coding agent step by step.
Each chapter adds one capability. The notebook IS the source of truth.

## Entry point
- `pocket_agent_colab.ipynb` — the entire project lives here

## Chapter guide
| Chapter | Topic | Key concepts defined |
|---------|-------|----------------------|
| 1 | Setup | `chat()`, `show_panel()`, `ping_ollama()` |
| 2 | Hello LLM | `count_tokens()`, `scan_repo_costs()` |
| 3 | Manifest | `load_manifest()`, `ask_with_manifest()` |
| 4 | Glob + JIT reads | `glob_files()`, `jit_read()` |
| 5 | Fuzzy scoring | `score_files()` |
| 6 | Grep | `grep_repo()` |
| 7 | Microcompaction | `compact()`, hot/cold store |
| 8 | Semantic RAG | `embed()`, `retrieve()` |
| 9 | Full pipeline | `run()` |
| 10 | Write + diff | `write_file()`, `make_diff()` |
| 11 | Agent loop | `agent_loop()` |
| 12 | Test generation | `generate_tests()` |

## Off-limits (never load these)
- `.git/` — git internals
- `__pycache__/` — compiled bytecode
- `*.ipynb_checkpoints/` — Jupyter autosave noise

## Questions about token budgeting → Chapter 3
## Questions about retrieval strategy → Chapters 4–8
## Questions about the agent loop → Chapter 11

### 4.2 `load_manifest()` — read the map

`load_manifest()` does two things:
1. Reads `AGENTS.md` as plain text (to inject into the prompt)
2. Extracts any file paths mentioned in it (for later retrieval stages to use as hints)

The path extraction is intentionally simple — a regex that finds things
that look like `path/to/file.ext`. False positives are fine; this is
advisory, not authoritative.

In [ ]:
import re

MANIFEST_FILENAME = "AGENTS.md"

def load_manifest(repo_root: str = REPO_ROOT) -> dict:
    """
    Read AGENTS.md from *repo_root*.

    Returns a dict:
        {
          "text":   str,        # full file content, ready to inject into a prompt
          "tokens": int,        # estimated token cost
          "paths":  list[str],  # file paths mentioned in the manifest
          "found":  bool,       # False if the file doesn't exist
        }
    """
    manifest_path = Path(repo_root) / MANIFEST_FILENAME

    if not manifest_path.exists():
        return {"text": "", "tokens": 0, "paths": [], "found": False}

    text = manifest_path.read_text(errors="ignore")

    # Extract things that look like file paths: word chars, slashes, dots
    # e.g.  pocket_agent.ipynb  src/parser.py  docs/architecture.md
    paths = re.findall(r'\b[\w./\-]+\.[\w]+\b', text)
    # Deduplicate while preserving order
    seen, unique_paths = set(), []
    for p in paths:
        if p not in seen:
            seen.add(p)
            unique_paths.append(p)

    return {
        "text":   text,
        "tokens": count_tokens(text),
        "paths":  unique_paths,
        "found":  True,
    }

manifest = load_manifest()
print(f"Manifest found : {manifest['found']}")
print(f"Token cost     : {manifest['tokens']}  ({manifest['tokens']/TOKEN_BUDGET*100:.1f}% of budget)")
print(f"Paths mentioned: {manifest['paths'][:8]}")

### 4.3 `ask_with_manifest()` — the manifest-aware query

Every prompt the agent sends from now on follows this structure:

```
[SYSTEM]
You are a coding assistant. Here is the project map:
<AGENTS.md contents>

[USER]
<query>
```

The manifest is injected once, costs a fixed number of tokens, and gives
the model the project layout before it has to reason about anything.

In [ ]:
def ask_with_manifest(
    query:     str,
    repo_root: str = REPO_ROOT,
    files:     list[dict] | None = None,
) -> tuple[str, int]:
    """
    Send *query* to the LLM with the project manifest prepended as a system prompt.

    Parameters
    ----------
    query     : the user's question
    repo_root : where to find AGENTS.md
    files     : already-loaded file dicts {"path", "content", "hot"}
                their content is appended after the manifest

    Returns (reply_text, tokens_used).
    """
    manifest = load_manifest(repo_root)
    files    = files or []

    # Build system prompt
    if manifest["found"]:
        system_content = (
            "You are a coding assistant with access to the project map below.\n"
            "Use it to understand the codebase structure before answering.\n\n"
            f"--- PROJECT MAP (AGENTS.md) ---\n{manifest['text']}\n---"
        )
    else:
        system_content = "You are a coding assistant."

    # Append any loaded file contents
    file_block = ""
    for f in files:
        file_block += f"\n\n--- FILE: {f['path']} ---\n{f.get('content', '')}"

    messages = [
        {"role": "system",  "content": system_content},
        {"role": "user",    "content": query + file_block},
    ]

    reply, tokens_used = chat(messages)
    return reply, tokens_used

### 4.4 Try it

Ask a question about the project. The manifest is in the prompt so the model
already knows the chapter structure before it replies.

In [ ]:
query = "Which chapter should I look at to understand how retrieval works?"

show_rule("Chapter 4 — manifest-guided query")
show_panel(
    query       = query,
    token_used  = 0,
    strategy    = "manifest",
    prompt_size = count_tokens(query),
)
reply, tokens_used = ask_with_manifest(query)

show_rule("After call")
show_panel(
    query       = query,
    token_used  = tokens_used,
    strategy    = "manifest",
    prompt_size = count_tokens(query),
)
show_reply(reply)


> **Pattern: load-once, reuse everywhere.**
>
> `load_manifest()` reads `AGENTS.md` once at startup and injects it into every
> subsequent prompt.  This is the *index-at-load-time* pattern: pay the I/O cost
> once, benefit on every query.  The same idea reappears in Chapter 9
> (`build_index()` caches embeddings in `_EMBED_INDEX`) and Chapter 12 (the
> manifest travels inside `plan_task()`'s system prompt without being re-read).


---
## Chapter 5 — Glob + JIT Reads

**Goal:** navigate a file tree without bulk-loading it.

Chapter 4 gave the agent a curated map. But what about files the map doesn't
mention, or repos where no `AGENTS.md` exists? The agent needs to *find* files
on its own — without reading them all.

The answer is a two-phase approach:

1. **Glob** — list every file that *could* be relevant (filenames only, no content).  
   This is essentially free: no tokens spent yet.
2. **JIT read** — load each file's content *just in time*, one at a time,  
   and stop when the token budget would tip past a safe threshold.

Files that were found but not loaded appear as **COLD** in the panel.
Files whose content is in the prompt are **HOT**.

**You will:**
- Create a small sample project to give glob something to work with
- Write `glob_files()` — match file paths against a pattern
- Write `jit_read()` — load one file on demand
- Write `budget_load()` — greedily load HOT files until budget threshold
- See the panel split between HOT and COLD for the first time

### 5.1 Sample project

The `sample_project/` files were already created back in section 3.3 — you should have them on disk already. If you skipped those cells, go back and run them now before continuing.

### 5.2 `glob_files()` — list without loading

`glob_files()` returns a list of matching file paths and their sizes —
but **no file content**. Zero tokens spent so far.

The `pattern` argument follows Python's `fnmatch` syntax:  
`*.py` matches any `.py` file, `utils/*.py` matches only files directly in `utils/`.

In [ ]:
import fnmatch

SKIP_DIRS = {".git", "__pycache__", "node_modules", ".venv", "venv",
             ".ipynb_checkpoints", ".mypy_cache", ".pytest_cache"}


def glob_files(
    pattern:   str,
    repo_root: str = REPO_ROOT,
) -> list[dict]:
    """
    Walk *repo_root* and return every file whose name matches *pattern*.

    Returns a list of dicts:
        {"path": str,   # relative to repo_root
         "bytes": int,  # file size — no content loaded
         "hot":   bool} # always False at this stage
    Sorted by path for deterministic ordering.
    """
    matches = []
    root = Path(repo_root)

    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [d for d in dirnames if d not in SKIP_DIRS and not d.startswith(".")]
        for fname in filenames:
            if fnmatch.fnmatch(fname, pattern):
                full = Path(dirpath) / fname
                try:
                    size = full.stat().st_size
                except OSError:
                    continue
                matches.append({
                    "path":  str(full.relative_to(root)),
                    "bytes": size,
                    "hot":   False,
                })

    return sorted(matches, key=lambda x: x["path"])

In [ ]:
# ── Demo ────────────────────────────────────────────────────────────────────
found = glob_files("*.py", repo_root="sample_project")
print(f"Found {len(found)} Python files:\n")
for f in found:
    print(f"  {f['path']:45s}  {f['bytes']:>5} bytes")

### 5.3 `jit_read()` — load one file on demand

`jit_read()` takes a path from the glob list and reads its content.
It returns the same dict shape, extended with `"content"` and `"tokens"`,
and flips `"hot"` to `True`.

In [ ]:
def jit_read(file_meta: dict, repo_root: str = REPO_ROOT) -> dict:
    """
    Read the content of one file described by *file_meta* (a glob result dict).

    Returns a new dict with the same keys plus:
        "content": str   — full file text
        "tokens":  int   — estimated token cost
        "hot":     True  — this file is now in the prompt
    """
    full_path = Path(repo_root) / file_meta["path"]
    try:
        # errors="ignore" silently skips bytes that aren't valid UTF-8.
        # Without it, reading a binary file (or a file saved in Latin-1)
        # would raise a UnicodeDecodeError and crash the whole retrieval pass.
        content = full_path.read_text(errors="ignore")
    except OSError as exc:
        content = f"[error reading file: {exc}]"

    return {
        **file_meta,          # preserve path, bytes, hot, score, etc.
        "content": content,
        "tokens":  count_tokens(content),
        "hot":     True,
    }

In [ ]:
# ── Demo: read one file ─────────────────────────────────────────────────────
sample_meta = glob_files("parser.py", repo_root="sample_project")[0]
loaded      = jit_read(sample_meta, repo_root="sample_project")

print(f"Path   : {loaded['path']}")
print(f"Tokens : {loaded['tokens']}")
print(f"Hot    : {loaded['hot']}")
print(f"\nFirst 3 lines:\n{''.join(loaded['content'].splitlines(keepends=True)[:3])}")

### 5.4 `budget_load()` — greedy HOT/COLD split

Now we combine glob and JIT read.
`budget_load()` takes a list of glob results, reads them one by one,
and stops before the token budget hits a safety threshold.
Files it couldn't fit are kept in the list as COLD.

The threshold is `0.7` (70 % of budget) — leaving room for the manifest,
the query, and the model's reply.


In [ ]:
HOT_THRESHOLD = 0.7   # stop loading when prompt would exceed 70 % of budget


def budget_load(
    candidates: list[dict],
    already_used: int      = 0,
    repo_root:    str      = REPO_ROOT,
    threshold:    float    = HOT_THRESHOLD,
) -> list[dict]:
    """
    JIT-read files from *candidates* until adding the next one would push
    the running token total past *threshold* × TOKEN_BUDGET.

    Returns the full candidate list with:
      - loaded files marked  hot=True  and populated with "content"/"tokens"
      - unloaded files kept  hot=False (no content key added)
    """
    budget_limit = int(TOKEN_BUDGET * threshold)
    used         = already_used
    result       = []

    for meta in candidates:
        headroom = budget_limit - used
        # Peek at size without reading: use byte count as a cheap proxy
        estimated = meta["bytes"] // 4
        if estimated > headroom:
            result.append({**meta, "hot": False})
            continue

        loaded = jit_read(meta, repo_root=repo_root)
        if used + loaded["tokens"] > budget_limit:
            result.append({**meta, "hot": False})
        else:
            used += loaded["tokens"]
            result.append(loaded)

    return result


In [ ]:
# ── Demo: load sample_project with a tiny artificial budget ─────────────────
# Set threshold so roughly the first half of files (by size) fit.
candidates   = glob_files("*.py", repo_root="sample_project")
total_tokens = sum(c["bytes"] // 4 for c in candidates)
# Target: load ~40% of the total so some files are hot, some cold
TINY_BUDGET  = (total_tokens * 0.40) / TOKEN_BUDGET
loaded_files = budget_load(candidates, repo_root="sample_project", threshold = TINY_BUDGET)

hot  = [f for f in loaded_files if f["hot"]]
cold = [f for f in loaded_files if not f["hot"]]

show_rule("budget_load() result")
show_panel(
    query        = "How does the project handle nested JSON objects?",
    token_used   = sum(f.get("tokens", 0) for f in hot),
    files        = [{"path": f["path"], "hot": f["hot"]} for f in loaded_files],
    strategy     = "glob + JIT",
    prompt_size  = sum(f.get("tokens", 0) for f in hot),
)
print(f"\nHOT ({len(hot)} files, {sum(f['tokens'] for f in hot):,} tokens):")
for f in hot:
    print(f"  ✓ {f['path']}")
print(f"\nCOLD ({len(cold)} files — found but not loaded):")
for f in cold:
    print(f"  ✗ {f['path']}")

### 5.5 Try it end-to-end

Ask a question about the sample project.
The agent globs for Python files, JIT-loads what fits in the budget,
and sends the HOT files plus the manifest to Ollama.

In [ ]:
query      = "How does the parser handle a JSON array of plain numbers like [1, 2, 3]?"
repo       = "sample_project"

# Phase 1: glob — free, no tokens spent
candidates   = glob_files("*.py", repo_root=repo)

# Phase 2: budget-aware JIT load
manifest     = load_manifest(repo)
manifest_tok = manifest["tokens"]
loaded_files = budget_load(candidates, already_used=manifest_tok, repo_root=repo)

hot_files    = [f for f in loaded_files if f["hot"]]
prompt_size  = manifest_tok + sum(f["tokens"] for f in hot_files) + count_tokens(query)

show_rule("Chapter 5 — before LLM call")
show_panel(
    query       = query,
    token_used  = prompt_size,
    files       = [{"path": f["path"], "hot": f["hot"]} for f in loaded_files],
    strategy    = "glob + JIT",
    prompt_size = prompt_size,
)
reply, tokens_used = ask_with_manifest(query, repo_root=repo, files=hot_files)

show_rule("After call")
show_panel(
    query       = query,
    token_used  = tokens_used,
    files       = [{"path": f["path"], "hot": f["hot"]} for f in loaded_files],
    strategy    = "glob + JIT",
    prompt_size = prompt_size,
)
show_reply(reply)


---
## Chapter 6 — Fuzzy Scoring

**Goal:** rank retrieved files before loading them, so budget_load() always drops the *least* relevant files when it runs out of room.

Chapter 5's glob returns files in alphabetical order.
If the budget fills up, it drops whatever comes last alphabetically — which may be the most relevant file.
Fuzzy scoring fixes this by sorting candidates by relevance *before* the JIT load loop.

Scoring works entirely on **file paths** — no content is read.
That keeps it free (no tokens, no disk I/O beyond what glob already did).

**Signals used:**
| Signal | Weight |
|--------|--------|
| Exact query word in filename stem | high |
| Fuzzy match between query word and filename stem | medium |
| Query word appears in a directory component | low |
| File is a test file, query doesn't mention tests | penalty ×1.5 |

**You will:**
- Write `tokenize_query()` — normalise a query into scoreable terms
- Write `score_file()` — score one file against the term list
- Write `rank_files()` — sort a candidate list by score descending
- Replace the alphabetical glob order with ranked order and see the panel change

### 6.1 `tokenize_query()` — extract scoreable terms

We strip stop words and short tokens so scores aren't diluted by
words like "the", "a", "how", "does".

In [ ]:
import difflib

_STOP_WORDS = {
    "a", "an", "the", "is", "in", "it", "of", "to", "do", "does",
    "how", "what", "why", "when", "where", "which", "who", "for",
    "and", "or", "but", "not", "with", "this", "that", "are", "was",
    "i", "me", "my", "we", "our", "you", "your",
}


def tokenize_query(query: str) -> list[str]:
    """
    Lowercase, split on non-alphanumeric chars, remove stop words and
    tokens shorter than 3 characters.

    "How does the formatter handle missing fields?"
    → ["formatter", "handle", "missing", "fields"]
    """
    words = re.split(r"[^a-zA-Z0-9]+", query.lower())
    return [w for w in words if len(w) >= 3 and w not in _STOP_WORDS]


In [ ]:
# Quick check
print(tokenize_query("How does the formatter handle missing fields?"))
print(tokenize_query("Where is the CSV delimiter configured?"))

### 6.2 `score_file()` — score one file against a query

`difflib.SequenceMatcher` gives us a similarity ratio between 0 and 1
with no extra dependencies — it's in the Python standard library.

In [ ]:
def score_file(meta: dict, query_terms: list[str]) -> float:
    """
    Score *meta* (a glob result dict) against *query_terms*.

    Higher = more relevant.  Returns 1.0 for an empty term list.
    No file content is read — scoring is path-only.
    """
    # Match weights — stem matters more than directory
    W_STEM_EXACT   = 4.0
    W_STEM_SUBSTR  = 2.5
    W_STEM_FUZZY   = 2.0  # multiplied by SequenceMatcher ratio (0–1)
    W_DIR_EXACT    = 2.0
    W_DIR_SUBSTR   = 1.5
    W_DIR_FUZZY    = 1.3  # multiplied by SequenceMatcher ratio (0–1)
    W_TEST_PENALTY = 0.5  # down-rank test files when query isn't about tests

    if not query_terms:
        return 1.0

    path  = meta["path"].lower()
    parts = Path(path).parts          # ("utils", "parser.py")
    stem  = Path(path).stem           # "parser"
    dirs  = parts[:-1]                # ("utils",)

    score = 1.0
    for term in query_terms:
        # ── Filename stem ────────────────────────────────────────────
        if term == stem:
            score += W_STEM_EXACT
        elif term in stem or stem in term:
            score += W_STEM_SUBSTR
        else:
            score += difflib.SequenceMatcher(None, term, stem).ratio() * W_STEM_FUZZY

        # ── Directory components ──────────────────────────────────────
        for d in dirs:
            if term == d:
                score += W_DIR_EXACT
            elif term in d:
                score += W_DIR_SUBSTR
            else:
                score += difflib.SequenceMatcher(None, term, d).ratio() * W_DIR_FUZZY

    # ── Test-file penalty ─────────────────────────────────────────────
    if "test" in path and "test" not in query_terms:
        score *= W_TEST_PENALTY

    return round(score, 4)

In [ ]:
# ── Sanity check ─────────────────────────────────────────────────────────────
_candidates = glob_files("*.py", repo_root="sample_project")
_terms      = tokenize_query("How does the formatter handle missing fields?")

for f in _candidates:
    print(f"{score_file(f, _terms):.4f}  {f['path']}")

### 6.3 `rank_files()` — sort the candidate list

A thin wrapper that applies `score_file()` to every candidate and
returns them sorted highest-score-first.
Files with a score of 0 are kept — they're still candidates,
just lowest priority.

In [ ]:
def rank_files(candidates: list[dict], query: str) -> list[dict]:
    """
    Return *candidates* sorted by relevance to *query*, highest first.
    Adds a "score" key to each dict.
    """
    terms = tokenize_query(query)
    scored = [
        {**c, "score": score_file(c, terms)}
        for c in candidates
    ]
    return sorted(scored, key=lambda x: x["score"], reverse=True)


In [ ]:
# ── Show ranked order vs alphabetical ────────────────────────────────────────
query      = "How does the formatter handle missing fields?"
candidates = glob_files("*.py", repo_root="sample_project")
ranked     = rank_files(candidates, query)

rows = ""
for i, f in enumerate(ranked, 1):
    color = "#2da44e" if i == 1 else ("#bf8700" if i <= 3 else "#8c959f")
    rows += (
        f'<tr><td style="text-align:right;padding:3px 10px;color:#8c959f">{i}</td>'
        f'<td style="text-align:right;padding:3px 10px;color:{color};font-weight:bold">{f["score"]:.4f}</td>'
        f'<td style="padding:3px 10px;font-family:monospace">{f["path"]}</td></tr>'
    )

display(HTML(f"""
<div style="font-size:0.88rem">
  <b>Ranked candidates</b>
  <table style="border-collapse:collapse;margin-top:6px">
    <tr style="border-bottom:1px solid #d0d7de;color:#57606a">
      <th style="text-align:right;padding:3px 10px">Rank</th>
      <th style="text-align:right;padding:3px 10px">Score</th>
      <th style="text-align:left;padding:3px 10px">File</th>
    </tr>
    {rows}
  </table>
</div>
"""))


### 6.4 Try it — ranked glob + JIT

Same query as Chapter 5, but now candidates are ranked before budget_load()
sees them. The first file that turns COLD will be the least relevant, not a
random alphabetical casualty.

In [ ]:
query = "How does the formatter handle missing fields?"
repo  = "sample_project"

# ── Temporarily shrink the budget so the demo shows HOT and COLD files ────────
# Our tiny sample project fits entirely in the real TOKEN_BUDGET, so
# budget_load marks everything HOT and ranking has no visible effect.
# Capping to 1,500 tokens forces budget_load to load only the top-ranked
# files (HOT) and leave the rest unloaded (COLD), which is the whole point.
# The real budget is restored at the end of the cell.
_saved_budget = TOKEN_BUDGET
TOKEN_BUDGET  = 1_500   # ← demo cap only

manifest     = load_manifest(repo)
manifest_tok = manifest["tokens"]

# We scope to source files only — test files happen to contain the word
# "formatter" too, which would pollute the ranking and obscure the lesson.
source_files = glob_files("*.py", repo_root=repo)

# ── Without ranking: glob order (arbitrary) ───────────────────────────────────
unranked_load = budget_load(source_files, already_used=manifest_tok, repo_root=repo)
unranked_hot  = [f for f in unranked_load if f["hot"]]
unranked_tok  = sum(f.get("tokens", 0) for f in unranked_hot)

# ── With ranking: best match first ───────────────────────────────────────────
ranked        = rank_files(source_files, query)
ranked_load   = budget_load(ranked, already_used=manifest_tok, repo_root=repo)
ranked_hot    = [f for f in ranked_load if f["hot"]]
ranked_cold   = [f for f in ranked_load if not f["hot"]]
ranked_tok    = sum(f.get("tokens", 0) for f in ranked_hot)

# ── Ask with the ranked HOT files ─────────────────────────────────────────────
prompt_size = manifest_tok + ranked_tok + count_tokens(query)
show_rule("Chapter 6 — fuzzy-ranked before LLM call")
show_panel(
    query       = query,
    token_used  = prompt_size,
    files       = [{"path": f["path"], "hot": f["hot"]} for f in ranked_load],
    strategy    = "glob + fuzzy + JIT",
    prompt_size = prompt_size,
)
reply, tokens_used = ask_with_manifest(query, repo_root=repo, files=ranked_hot)

show_rule("After call")
show_panel(
    query       = query,
    token_used  = tokens_used,
    files       = [{"path": f["path"], "hot": f["hot"]} for f in ranked_load],
    strategy    = "glob + fuzzy + JIT",
    prompt_size = prompt_size,
)
show_reply(reply)

# ── Restore the real budget ───────────────────────────────────────────────────
TOKEN_BUDGET = _saved_budget


---
## Chapter 7 — Grep

**Goal:** find files by what's *inside* them, not just their name.

Chapters 4–5 rank files by path relevance. That works well when the
query mentions a filename — "how does the *formatter* work?" — but fails
when the relevant file has a generic name.

Consider: *"Where does the code raise a TypeError?"*
No filename contains "typeerror". Fuzzy scoring gives every file a near-zero
path score. But `validator.py` *contains* `raise TypeError(...)` — and grep
finds it instantly.

Grep is the third retrieval strategy. It searches file content for
patterns derived from the query, counts matches per file, and uses that
count as an additional relevance signal on top of the fuzzy path score.

**You will:**
- Write `grep_repo()` — search file contents, return matches with excerpts
- Write `query_to_patterns()` — turn query terms into search patterns
- Write `grep_rank()` — combine grep hit count with fuzzy path score
- See grep surface `validator.py` for a query that fuzzy scoring misses entirely

### 7.1 `grep_repo()` — search file contents

For each file, we search for a compiled regex pattern and collect:
- the number of matching lines (used for ranking)
- up to `context_lines` surrounding each match (used in the prompt as an excerpt)

Returning excerpts rather than full file content is a key budget trick:
if grep finds 2 matching lines in a 300-line file, we can show just those
2 lines plus context instead of loading all 300 lines HOT.

In [ ]:
def grep_repo(
    pattern:       str,
    repo_root:     str   = REPO_ROOT,
    extensions:    set   = CODE_EXTENSIONS,
    context_lines: int   = 2,
    max_matches:   int   = 5,
) -> list[dict]:
    """
    Search every matching file under *repo_root* for *pattern* (regex).

    Returns a list of dicts — one per file that had at least one match:
        {
          "path":      str,         # relative to repo_root
          "hit_count": int,         # number of matching lines
          "excerpt":   str,         # up to max_matches hits with context
          "bytes":     int,
          "hot":       False,       # budget_load() will flip this
          "score":     1.0,         # grep_rank() will fill this
        }
    Sorted by hit_count descending.
    """
    try:
        regex = re.compile(pattern, re.IGNORECASE)
    except re.error:
        regex = re.compile(re.escape(pattern), re.IGNORECASE)

    results = []
    root    = Path(repo_root)

    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [d for d in dirnames if d not in SKIP_DIRS and not d.startswith(".")]
        for fname in filenames:
            full = Path(dirpath) / fname
            if full.suffix.lower() not in extensions:
                continue
            try:
                lines = full.read_text(errors="ignore").splitlines()
            except OSError:
                continue

            # Find matching line indices
            hit_indices = [i for i, ln in enumerate(lines) if regex.search(ln)]
            if not hit_indices:
                continue

            # Build excerpt: up to max_matches hits with context
            excerpts, seen_lines = [], set()
            for hit_i in hit_indices[:max_matches]:
                start = max(0, hit_i - context_lines)
                end   = min(len(lines), hit_i + context_lines + 1)
                block = []
                for ln_i in range(start, end):
                    if ln_i not in seen_lines:
                        prefix = "→ " if ln_i == hit_i else "  "
                        block.append(f"{prefix}{ln_i+1:4d} │ {lines[ln_i]}")
                        seen_lines.add(ln_i)
                excerpts.append("\n".join(block))

            excerpt_text = "\n\n".join(excerpts)

            results.append({
                "path":      str(full.relative_to(root)),
                "hit_count": len(hit_indices),
                "excerpt":   excerpt_text,
                "bytes":     full.stat().st_size,
                "hot":       False,
                "score":     1.0,
            })

    return sorted(results, key=lambda x: x["hit_count"], reverse=True)


In [ ]:
hits = grep_repo("TypeError", repo_root="sample_project")
for h in hits:
    print(f"{h['hit_count']} hit(s)  {h['path']}")
    print(h["excerpt"])
    print()


### 7.2 `query_to_patterns()` — derive search patterns from a query

We turn the query terms into a single alternation regex:
`formatter|missing|fields`

This is a liberal match — any file mentioning *any* term is a candidate.
We'll use hit count to separate strong matches from weak ones.

In [ ]:
def query_to_patterns(query: str) -> str:
    """
    Convert a natural-language query into a single regex pattern
    suitable for grep_repo().

    "Where does the code raise a TypeError?"
    → "typeerror|raise|code"   (terms ≥ 4 chars, lowercased, joined with |)

    Returns an empty string if no usable terms are found.
    """
    terms = [t for t in tokenize_query(query) if len(t) >= 4]
    if not terms:
        return ""
    return "|".join(re.escape(t) for t in terms)


# Check a few examples
for q in [
    "Where does the code raise a TypeError?",
    "How does the formatter handle missing fields?",
    "What is the CSV delimiter default value?",
]:
    print(f"  {q!r}\n  → {query_to_patterns(q)!r}\n")

### 7.3 `grep_rank()` — combine grep hits with fuzzy path score

A file can score high two ways:
- Many grep hits → strong content signal
- High fuzzy path score → strong name signal

`grep_rank()` normalises both signals to [0, 1] and adds them,
so a file with a relevant name *and* relevant content beats one with only one signal.

In [ ]:
def grep_rank(
    query:     str,
    repo_root: str = REPO_ROOT,
) -> list[dict]:
    """
    Full grep-based retrieval pipeline for *query*:
    1. Derive a regex pattern from the query terms
    2. Grep every source file for that pattern
    3. Score each hit file with both grep hit_count and fuzzy path score
    4. Normalise both signals to [0, 1] and sum them
    5. Return sorted by combined score descending
    Files with zero grep hits are excluded entirely.
    """
    pattern = query_to_patterns(query)
    if not pattern:
        return []
    hits = grep_repo(pattern, repo_root=repo_root)
    if not hits:
        return []
    terms          = tokenize_query(query)
    max_hits       = max(h["hit_count"] for h in hits) or 1
    max_path_score = max(score_file(h, terms) for h in hits) or 1
    ranked = []
    for h in hits:
        norm_grep = h["hit_count"]    / max_hits
        norm_path = score_file(h, terms) / max_path_score
        combined  = round(norm_grep + norm_path, 4)
        ranked.append({**h, "score": combined})
    return sorted(ranked, key=lambda x: x["score"], reverse=True)


In [ ]:
# ── Demo ──────────────────────────────────────────────────────────────────────
query  = "Where does the code raise a TypeError?"
ranked = grep_rank(query, repo_root="sample_project")

rows = ""
for f in ranked:
    excerpt = (f["excerpt"].splitlines()[0] if f["excerpt"] else "").replace("<","&lt;")
    rows += (
        f'<tr>'
        f'<td style="text-align:right;padding:3px 10px;color:#2da44e;font-weight:bold">{f["score"]}</td>'
        f'<td style="text-align:right;padding:3px 10px">{f["hit_count"]}</td>'
        f'<td style="padding:3px 10px;font-family:monospace">{f["path"]}</td>'
        f'<td style="padding:3px 10px;font-family:monospace;color:#57606a;font-size:0.82rem">{excerpt}</td>'
        f'</tr>'
    )

display(HTML(f"""
<div style="font-size:0.88rem">
  <b>grep_rank(): {query!r}</b>
  <table style="border-collapse:collapse;margin-top:6px">
    <tr style="border-bottom:1px solid #d0d7de;color:#57606a">
      <th style="text-align:right;padding:3px 10px">Score</th>
      <th style="text-align:right;padding:3px 10px">Hits</th>
      <th style="text-align:left;padding:3px 10px">File</th>
      <th style="text-align:left;padding:3px 10px">First match</th>
    </tr>
    {rows}
  </table>
</div>
"""))

fuzzy_only = rank_files(glob_files("*.py", repo_root="sample_project"), query)
print("\nFuzzy path scores for same query (for comparison):")
for f in fuzzy_only:
    print(f"  {f['score']:.4f}  {f['path']}")


### 7.4 Try it — grep-guided query with excerpt injection

One more trick: instead of loading the *full* file HOT, we can inject
just the grep **excerpt** for files that matched well but are expensive
to load in full. This stretches the budget further.

For this demo we load full content (the sample files are small), but
the `excerpt` field is there and Chapter 8 will start using it during compaction.

In [ ]:
query = "Where does the code raise a TypeError?"
repo  = "sample_project"

# Grep-ranked candidates — only files that contain a matching line
candidates = grep_rank(query, repo_root=repo)

# Fall back to fuzzy glob for files with zero grep hits
all_paths   = {f["path"] for f in candidates}
glob_extras = [f for f in rank_files(glob_files("*.py", repo_root=repo), query)
               if f["path"] not in all_paths]
candidates  = candidates + glob_extras   # grep hits first, fuzzy extras after

# Budget-aware JIT load
manifest     = load_manifest(repo)
manifest_tok = manifest["tokens"]
loaded_files = budget_load(candidates, already_used=manifest_tok, repo_root=repo)

hot_files   = [f for f in loaded_files if f["hot"]]
prompt_size = manifest_tok + sum(f["tokens"] for f in hot_files) + count_tokens(query)

show_rule("Chapter 7 — grep-ranked before LLM call")
show_panel(
    query       = query,
    token_used  = prompt_size,
    files       = [{"path": f["path"], "hot": f["hot"]} for f in loaded_files],
    strategy    = "grep + fuzzy + JIT",
    prompt_size = prompt_size,
)
reply, tokens_used = ask_with_manifest(query, repo_root=repo, files=hot_files)

show_rule("After call")
show_panel(
    query       = query,
    token_used  = tokens_used,
    files       = [{"path": f["path"], "hot": f["hot"]} for f in loaded_files],
    strategy    = "grep + fuzzy + JIT",
    prompt_size = prompt_size,
)
show_reply(reply)


---
## Chapter 8 — Microcompaction

**Goal:** survive long sessions without hitting the context ceiling.

After several turns the conversation grows: loaded file contents, prior replies,
system prompts. Eventually the budget fills up and the model starts forgetting
the beginning of the conversation.

Microcompaction solves this with a simple rule:

> When the running token count exceeds a **compaction threshold**,
> take the coldest HOT files, summarise each one in a single LLM call,
> replace the full content with the summary, and mark them COLD.

The summary is much smaller than the original — typically 10–15 % of the
original token count. Budget is freed; the session continues.

```
Before compaction            After compaction
─────────────────            ────────────────
HOT  utils/parser.py  420t   COLD utils/parser.py  [summary: 45t]
HOT  utils/formatter  380t   HOT  utils/formatter  380t   ← kept
HOT  utils/validator  290t   HOT  utils/validator  290t   ← kept
─────────────────            ────────────────
total: 1090t                 total: 715t   (saved 375t)
```

**You will:**
- Write `eviction_candidates()` — pick which HOT files to evict
- Write `summarise_file()` — compress one file to a short summary via Ollama
- Write `compact()` — orchestrate eviction + summarisation, return updated file list
- Simulate a session running over budget and watch compaction kick in

### 8.1 `eviction_candidates()` — which HOT files to evict first

We evict the files that are least likely to be needed again.
The heuristic: **lowest score first** (from Chapter 6's fuzzy scoring).
Files with no score get evicted before files that matched the current query.

We also always keep at least one HOT file — evicting everything defeats the purpose.

In [ ]:
COMPACT_THRESHOLD = 0.80   # trigger compaction when prompt > 80 % of budget
EVICT_TARGET      = 0.55   # compact down to 55 % of budget


def eviction_candidates(
    files:  list[dict],
    query:  str,
    n_keep: int = 1,
) -> tuple[list[dict], list[dict]]:
    """
    Split HOT files into (evict, keep) by relevance score.
    Lowest-scoring HOT files are evicted first; top *n_keep* are retained.
    COLD files are not touched — they're never in the prompt to begin with.

    Returns (to_evict, to_keep) — both lists contain only HOT files.
    """
    hot = [f for f in files if f.get("hot")]

    if len(hot) <= n_keep:
        return [], hot

    terms  = tokenize_query(query)
    scored = sorted(hot, key=lambda f: f.get("score", score_file(f, terms)))

    return scored[:-n_keep], scored[-n_keep:]


# ── Quick check ──────────────────────────────────────────────────────────────
_demo_files = [
    {"path": "utils/parser.py",    "hot": True,  "score": 1.8,  "tokens": 420},
    {"path": "utils/formatter.py", "hot": True,  "score": 3.1,  "tokens": 380},
    {"path": "utils/validator.py", "hot": True,  "score": 1.3,  "tokens": 290},
    {"path": "main.py",            "hot": False, "score": 1.1,  "tokens": 120},
]
evict, keep = eviction_candidates(_demo_files, "how does the formatter work?")
assert [f["path"] for f in evict] == ["utils/validator.py", "utils/parser.py"]
assert [f["path"] for f in keep]  == ["utils/formatter.py"]
print("eviction_candidates ✓")

### 8.2 `summarise_file()` — compress one file with the LLM

We ask the model for a terse technical summary in plain prose — no code blocks,
no bullet points. The goal is a 3–5 sentence description that preserves
the key exported names and their purpose, so later queries can still
reference this file even though its full content is no longer in the prompt.

In [ ]:
def summarise_file(file_dict: dict) -> dict:
    """
    Ask the LLM to compress *file_dict["content"]* to a short summary.

    Returns a new dict with:
      - "content"  replaced by the summary text
      - "tokens"   updated to the summary's token count
      - "hot"      set to False  (evicted from active prompt)
      - "summary"  set to True   (flag so later code knows this is compressed)
    """
    content = file_dict.get("content", "")
    if not content.strip():
        return {**file_dict, "hot": False, "summary": True}

    prompt = (
        f"Summarise the following source file in 3–5 sentences of plain prose. "
        f"Name the key functions/classes and what they do. "
        f"No code blocks, no bullet points.\n\n"
        f"FILE: {file_dict['path']}\n\n{content}"
    )
    summary_text, _ = chat([{"role": "user", "content": prompt}])

    return {
        **file_dict,
        "content": f"[SUMMARY of {file_dict['path']}]\n{summary_text}",
        "tokens":  count_tokens(summary_text),
        "hot":     False,
        "summary": True,
    }


In [ ]:
_parser_meta   = glob_files("parser.py", repo_root="sample_project")[0]
_parser_loaded = jit_read(_parser_meta, repo_root="sample_project")

print(f"Before: {_parser_loaded['tokens']} tokens")
_parser_summary = summarise_file(_parser_loaded)
print(f"After : {_parser_summary['tokens']} tokens  "
      f"({_parser_summary['tokens']/_parser_loaded['tokens']*100:.0f}% of original)\n")
print(_parser_summary["content"])


### 8.3 `compact()` — orchestrate the full compaction pass

`compact()` is called by the agent whenever the token count crosses
`COMPACT_THRESHOLD`. It loops through eviction candidates, summarises
each one, and returns the updated file list plus a log of what was freed.

In [ ]:
def compact(
    files:        list[dict],
    query:        str,
    token_used:   int,
    token_budget: int   = None,
    threshold:    float = COMPACT_THRESHOLD,
    evict_target: float = EVICT_TARGET,
) -> tuple[list[dict], int, list[str]]:
    """
    If *token_used* exceeds threshold × token_budget, evict and summarise
    the least-relevant HOT files until usage drops below evict_target × token_budget.

    token_budget defaults to TOKEN_BUDGET (the global context window).
    Pass a smaller value in demos/tests to trigger compaction on tiny file sets.

    Loop exits when either:
      - token usage drops below evict_target × token_budget, or
      - only n_keep HOT files remain (nothing left to evict)

    Returns:
        (updated_files, new_token_used, log_lines)
    where log_lines is a human-readable list of what was compacted.
    """
    budget    = token_budget if token_budget is not None else TOKEN_BUDGET
    ceiling   = int(budget * threshold)
    target    = int(budget * evict_target)

    if token_used <= ceiling:
        return files, token_used, []   # nothing to do

    log     = [f"Compaction triggered: {token_used:,} / {budget:,} tokens "
               f"({token_used/budget*100:.0f}%)"]
    updated = list(files)
    used    = token_used

    while used > target:
        to_evict, _ = eviction_candidates(updated, query, n_keep=1)
        if not to_evict:
            break

        victim = to_evict[0]
        before = victim.get("tokens", 0)

        summarised = summarise_file(victim)
        after      = summarised["tokens"]
        saved      = before - after

        updated = [summarised if f["path"] == victim["path"] else f
                   for f in updated]
        used   -= saved
        log.append(f"  compacted {victim['path']}: {before}t → {after}t  (saved {saved}t)")

    log.append(f"After compaction: {used:,} / {budget:,} tokens "
               f"({used/budget*100:.0f}%)")
    return updated, used, log

### 8.4 Simulation — watch compaction fire

We simulate a session that loads all sample files and artificially inflates
the token count past the compaction threshold, then watch `compact()` log
what it evicts and how much budget it recovers.

In [ ]:
query = "How does the formatter handle missing fields?"
repo  = "sample_project"

# Load all Python files HOT
all_files  = glob_files("*.py", repo_root=repo)
loaded     = [jit_read(f, repo_root=repo) for f in all_files]
terms      = tokenize_query(query)
loaded     = [{**f, "score": score_file(f, terms)} for f in loaded]
token_used = sum(f["tokens"] for f in loaded)

# Use a demo budget smaller than the real content so compaction fires on
# actual file summaries — no artificial padding needed.
demo_budget = int(token_used * 0.6)

show_rule("Before compaction")
show_panel(
    query        = query,
    token_used   = token_used,
    token_budget = demo_budget,
    files        = [{"path": f["path"], "hot": f["hot"]} for f in loaded],
    strategy     = "compact",
    prompt_size  = token_used,
)

# Run compaction against the small demo budget
updated, new_used, log = compact(loaded, query, token_used, token_budget=demo_budget)

show_rule("Compaction log")
for line in log:
    print(f"  {line}")

show_rule("After compaction")
show_panel(
    query        = query,
    token_used   = new_used,
    token_budget = demo_budget,
    files        = [{"path": f["path"], "hot": f["hot"]} for f in updated],
    strategy     = "compact",
    prompt_size  = new_used,
)

---
## Chapter 9 — Semantic RAG

**Goal:** retrieve files by meaning, not just keywords or filenames.

Grep and fuzzy scoring both rely on surface-level text overlap.
They miss cases like: *"where is the nesting logic handled?"* — the relevant
function is `_flatten` in `parser.py`, but neither the query nor the filename
contains the word "flatten".

Semantic retrieval fixes this by:
1. **Embedding** each file (or chunk) into a vector using `nomic-embed-text`
2. **Embedding** the query into the same vector space at runtime
3. **Ranking** files by cosine similarity — close vectors = similar meaning

The embeddings are computed once and cached in memory. On subsequent calls
only the (cheap) query embedding is recomputed.

**You will:**
- Write `embed()` — call Ollama's embedding endpoint
- Write `build_index()` — embed all files and store vectors in memory
- Write `semantic_retrieve()` — embed the query and return ranked files
- Compare semantic vs grep for a query that grep misses

### 9.1 Dependencies

Chapter 9 needs `numpy` for cosine similarity.

The embedding backend depends on your `EMBED_BACKEND` setting:

- **`"ollama"`** (local) — uses `nomic-embed-text` via the Ollama server. Make sure it's pulled: `ollama pull nomic-embed-text`
- **`"sentence-transformers"`** (Colab) — downloads `all-MiniLM-L6-v2` from Hugging Face on first run (~80 MB). Runs directly on Colab's CPU/GPU, no API calls needed. Takes a little time to get done, so be patient.

In [ ]:
import subprocess, sys, warnings, logging

_CH9_DEPS = ["numpy"]
if EMBED_BACKEND == "sentence-transformers":
    _CH9_DEPS.append("sentence-transformers")

for _pkg in _CH9_DEPS:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", _pkg],
        stdout=subprocess.DEVNULL,
    )

import numpy as np

# ── Load the embedding model for the chosen backend ───────────────────────────
_ST_MODEL = None   # only set when EMBED_BACKEND == "sentence-transformers"

if EMBED_BACKEND == "sentence-transformers":
    # Suppress HuggingFace auth warnings — we're using a public model,
    # no account or token needed.
    warnings.filterwarnings("ignore", message=".*HF_TOKEN.*")
    warnings.filterwarnings("ignore", message=".*unauthenticated.*")
    logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

    from sentence_transformers import SentenceTransformer
    # First run: downloads ~80 MB of weights from Hugging Face (cached afterwards)
    _ST_MODEL = SentenceTransformer("all-MiniLM-L6-v2")
    print(f"Embed backend  : sentence-transformers (all-MiniLM-L6-v2)")
    print(f"Vector dim     : {_ST_MODEL.get_sentence_embedding_dimension()}")
else:
    ok, models = ping_ollama()
    if ok and any(_ollama.embed in m for m in models):
        print(f"Embed backend  : ollama ({_ollama.embed})")
    else:
        print(f"WARNING: embed model '{_ollama.embed}' not found.")
        print(f"Run: ollama pull {_ollama.embed}")

print("Chapter 9 dependencies ready.")

### 9.2 `embed()` — call Ollama's embedding endpoint

Ollama exposes `/api/embed` (or `/api/embeddings` in older versions).
It accepts a model name and a string, returns a float vector.
We normalise the vector to unit length immediately so cosine similarity
later is just a dot product.

In [ ]:
def embed(text: str, model: str | None = None) -> np.ndarray:
    """
    Return a unit-normalised embedding vector for *text*.

    Backend is selected by the global EMBED_BACKEND variable:
      "sentence-transformers" — all-MiniLM-L6-v2 loaded in Chapter 9.1 (Colab default)
      "ollama"                — Ollama's /api/embed endpoint (local server)
    """
    if EMBED_BACKEND == "sentence-transformers":
        vec  = _ST_MODEL.encode([text], convert_to_numpy=True)[0]
        arr  = np.array(vec, dtype=np.float32)
        norm = np.linalg.norm(arr)
        return arr / norm if norm > 0 else arr

    # ── Ollama backend ────────────────────────────────────────────────────────
    m       = model or _ollama.embed
    payload = {"model": m, "input": text}
    try:
        r = requests.post(f"{_ollama.base_url}/api/embed", json=payload, timeout=60)
        r.raise_for_status()
        vec = r.json()["embeddings"][0]
    except Exception:
        # Fallback for older Ollama versions
        r = requests.post(f"{_ollama.base_url}/api/embeddings",
                          json={"model": m, "prompt": text}, timeout=60)
        r.raise_for_status()
        vec = r.json()["embedding"]

    arr  = np.array(vec, dtype=np.float32)
    norm = np.linalg.norm(arr)
    return arr / norm if norm > 0 else arr


In [ ]:
_v = embed("hello world")
print(f"Embed backend  : {EMBED_BACKEND}")
print(f"Vector dim     : {len(_v)}")
print(f"Vector norm    : {np.linalg.norm(_v):.6f}  (should be ~1.0)")

### 9.3 `build_index()` — embed every file, cache in memory

We chunk each file's content, embed every chunk, and store all the vectors
alongside the file metadata.  At query time we score a file by its *best*
matching chunk — so a relevant function buried at the end of a long file
still gets found.

For small files (like our sample project) one chunk covers the whole file,
so the behaviour is identical to a single embed call.  Cost scales linearly
with file size, not repo size.

> **Cost warning** — `build_index` makes one embedding call *per chunk per
> file*, and the result lives only in memory.  For a large repo with thousands
> of files this becomes slow and memory-hungry on every process restart.
>
> Production systems solve this with:
> - **Persistent vector stores** — ChromaDB, FAISS, pgvector, Weaviate —
>   embed once, persist to disk, query forever.
> - **Approximate nearest-neighbour (ANN) search** — sub-linear query time
>   instead of scanning every vector linearly.
> - **Orchestration libraries** — LlamaIndex, LangChain — handle chunking,
>   indexing, and retrieval behind a clean API.
>
> In Chapter 16 we rebuild this exact pipeline with LlamaIndex so you can
> see the 1-to-1 mapping between what we wrote here and what the library
> provides — and understand *why* the library exists rather than just
> cargo-culting it.

In [ ]:
# In-memory index: repo_root → list of {"path", "vectors", "bytes", "hot"}
_EMBED_INDEX: dict[str, list[dict]] = {}

CHUNK_SIZE    = 4000   # characters per chunk
CHUNK_OVERLAP = 200    # overlap between consecutive chunks


def embed_chunks(text: str) -> list[np.ndarray]:
    """
    Split *text* into overlapping chunks and embed each one.
    Returns a list of unit-norm vectors (one per chunk).
    Single-chunk files produce a one-element list — identical cost to before.
    """
    vecs  = []
    start = 0
    while start < len(text):
        chunk = text[start:start + CHUNK_SIZE]
        vec   = embed(chunk)
        if len(vec) > 0:
            vecs.append(vec)
        start += CHUNK_SIZE - CHUNK_OVERLAP
    return vecs


def build_index(
    repo_root:  str  = REPO_ROOT,
    extensions: set  = CODE_EXTENSIONS,
    force:      bool = REBUILD_INDEX,
) -> list[dict]:
    """
    Embed every source file under *repo_root* and cache the result.

    Each index entry stores *all* chunk vectors for the file.
    semantic_retrieve() scores by the best-matching chunk.

    Returns the index (list of dicts with a "vectors" key).
    On subsequent calls the cached index is returned unless *force=True*.
    """
    if repo_root in _EMBED_INDEX and not force:
        return _EMBED_INDEX[repo_root]

    files = glob_files("*", repo_root=repo_root)
    files = [f for f in files
             if Path(f["path"]).suffix.lower() in extensions]

    index = []
    for f in files:
        full_path = Path(repo_root) / f["path"]
        try:
            content = full_path.read_text(errors="ignore")
        except OSError:
            continue

        if not content.strip():
            continue

        vecs = embed_chunks(content)
        if not vecs:
            continue

        n_chunks = len(vecs)
        index.append({**f, "vectors": vecs, "content": content,
                      "tokens": count_tokens(content)})
        print(f"  embedded  {f['path']}  ({n_chunks} chunk{'s' if n_chunks > 1 else ''})")

    _EMBED_INDEX[repo_root] = index
    return index


In [ ]:
show_rule("Building semantic index for sample_project")
idx = build_index(repo_root="sample_project", force=True)
print(f"\nIndex contains {len(idx)} files.")

### 9.4 `semantic_retrieve()` — rank by cosine similarity

Cosine similarity between unit vectors is just a dot product.
We embed the query, dot it against every file vector, and sort descending.
The result dict shape matches what `budget_load()` and `show_panel()` expect.

In [ ]:
def semantic_retrieve(
    query:     str,
    repo_root: str = REPO_ROOT,
    top_k:     int = 5,
) -> list[dict]:
    """
    Return the top-*k* files from the semantic index, ranked by cosine
    similarity to *query*.

    Each returned dict has a "score" key (cosine similarity, 0–1).
    """
    index     = build_index(repo_root=repo_root)
    query_vec = embed(query)

    if len(query_vec) == 0:
        # Query embedding failed (e.g. empty query) — fall back to empty list
        return []

    scored = []
    for entry in index:
        # Score by best-matching chunk — a relevant section anywhere in the
        # file wins, not just the beginning.
        chunk_sims = [
            float(np.dot(query_vec, vec))
            for vec in entry.get("vectors", [entry.get("vector", np.array([]))])
            if len(vec) > 0 and vec.shape == query_vec.shape
        ]
        if not chunk_sims:
            continue
        sim = max(chunk_sims)
        scored.append({
            "path":    entry["path"],
            "bytes":   entry["bytes"],
            "tokens":  entry["tokens"],
            "content": entry["content"],
            "hot":     False,
            "score":   round(sim, 4),
        })

    return sorted(scored, key=lambda x: x["score"], reverse=True)[:top_k]


In [ ]:
# ── Demo: the query grep struggles with ──────────────────────────────────────
query = "Where is the nesting logic handled?"

sem_results  = semantic_retrieve(query, repo_root="sample_project")
grep_results = grep_rank(query, repo_root="sample_project")

rows = ""
for i, f in enumerate(sem_results, 1):
    rows += f'<tr><td>semantic</td><td style="text-align:right">{i}</td><td style="text-align:right;color:#2da44e">{f["score"]}</td><td><code>{f["path"]}</code></td></tr>'
for i, f in enumerate(grep_results, 1):
    rows += f'<tr><td>grep</td><td style="text-align:right">{i}</td><td style="text-align:right;color:#bf8700">{f["score"]}</td><td><code>{f["path"]}</code></td></tr>'

display(HTML(f"""
<table style="border-collapse:collapse;font-family:monospace;font-size:0.9rem;width:100%">
  <caption style="font-weight:bold;margin-bottom:6px">Semantic vs Grep</caption>
  <thead><tr style="border-bottom:2px solid #d0d7de">
    <th style="text-align:left;padding:4px 12px">Method</th>
    <th style="text-align:right;padding:4px 12px">Rank</th>
    <th style="text-align:right;padding:4px 12px">Score</th>
    <th style="text-align:left;padding:4px 12px">File</th>
  </tr></thead>
  <tbody>{rows}</tbody>
</table>
"""))


### 9.5 Try it end-to-end

Ask the nesting query. Semantic retrieval finds `parser.py` at the top
because `_flatten` is semantically close to "nesting logic",
even though neither word appears in the other.

In [ ]:
query = "Where is the nesting logic handled?"
repo  = "sample_project"

candidates   = semantic_retrieve(query, repo_root=repo)
manifest     = load_manifest(repo)
manifest_tok = manifest["tokens"]
loaded_files = budget_load(candidates, already_used=manifest_tok, repo_root=repo)

hot_files   = [f for f in loaded_files if f["hot"]]
prompt_size = manifest_tok + sum(f["tokens"] for f in hot_files) + count_tokens(query)

show_rule("Chapter 9 — semantic retrieval before LLM call")
show_panel(
    query       = query,
    token_used  = prompt_size,
    files       = [{"path": f["path"], "hot": f["hot"]} for f in loaded_files],
    strategy    = "semantic RAG",
    prompt_size = prompt_size,
)
reply, tokens_used = ask_with_manifest(query, repo_root=repo, files=hot_files)

show_rule("After call")
show_panel(
    query       = query,
    token_used  = tokens_used,
    files       = [{"path": f["path"], "hot": f["hot"]} for f in loaded_files],
    strategy    = "semantic RAG",
    prompt_size = prompt_size,
)
show_reply(reply)


> **Pattern: strategy dispatch.**
>
> `pick_strategy()` returns a string — `"semantic"`, `"grep"`, or `"budget"` —
> and `run()` uses it to choose which retrieval path to call.  This is the
> *strategy pattern*: the caller picks a named strategy; the dispatcher executes
> it.  You will add a fourth strategy in Exercise B (Chapter 14) without
> touching any of the existing three paths.


---
## Chapter 10 — Full Pipeline

**Goal:** one function call, `run(query, repo)`, does everything.

Every chapter so far introduced one tool. Chapter 10 wires them together:

```
query
  │
  ├─ manifest loaded (always)
  │
  ├─ strategy picker
  │     ├─ "semantic"  → semantic_retrieve()
  │     ├─ "grep"      → grep_rank()
  │     └─ "fuzzy"     → rank_files(glob_files())
  │
  ├─ budget_load()   — JIT read, HOT/COLD split
  │
  ├─ compact()       — if > COMPACT_THRESHOLD
  │
  └─ ask_with_manifest()  → reply
```

The **strategy picker** is a small classifier that looks at the query:
- Contains a specific symbol or quoted string → grep (looking for code)
- Vague / conceptual phrasing → semantic (understanding meaning)
- Everything else → fuzzy (fast, good enough for named-file queries)

**You will:**
- Write `pick_strategy()` — classify the query
- Write `run()` — the full pipeline in one call
- Try three queries that each route to a different strategy

### 10.1 `pick_strategy()` — classify the query

A lightweight heuristic — no ML needed.
The three signals are mutually exclusive and checked in priority order.

In [ ]:
# Patterns that suggest the user is looking for a specific symbol or string
_GREP_SIGNALS = re.compile(
    r'(?:'
    r'"[^"]+"'
    r'|`[^`]+`'
    r'|raise\s+\w+'
    r'|def\s+\w+'
    r'|class\s+\w+'
    r'|import\s+\w+'
    r'|\b[A-Z][a-zA-Z]+Error\b'
    r')'
)

# Words that suggest the user is asking about a concept, not looking for a symbol
# Deliberately narrow — "what" and "how" are too generic and cause false positives
_CONCEPTUAL_WORDS = {
    "why", "explain", "design", "architecture",
    "pattern", "approach", "concept", "idea", "strategy", "logic",
    "purpose", "reason", "difference", "relationship",
}

def pick_strategy(query: str) -> str:
    """
    Classify *query* into one of three retrieval strategies:
      "grep"     — looking for a specific symbol / string in code
      "semantic" — conceptual / meaning-based question
      "fuzzy"    — everything else (filename-level search)
    """
    # Grep signal: backticks, quotes, error names, def/class/import
    if _GREP_SIGNALS.search(query):
        return "grep"

    # Semantic signal: conceptual vocabulary
    words = set(query.lower().split())
    if words & _CONCEPTUAL_WORDS:
        return "semantic"

    return "fuzzy"


In [ ]:
for q in [
    "Where does the code raise a `TypeError`?",           # → grep
    "Why does the parser wrap scalars in a dict?",        # → semantic
    "What does the formatter do with the delimiter?",     # → fuzzy
]:
    print(f"  {pick_strategy(q):8s}  {q}")

### 10.2 `run()` — the full pipeline

This is the function all later chapters call.
It returns a `RunResult` named tuple so callers can inspect
the files that were loaded, the strategy used, and the token counts.

In [ ]:
from typing import NamedTuple


class RunResult(NamedTuple):
    reply:       str
    strategy:    str
    files:       list[dict]   # full list, HOT and COLD
    tokens_used: int
    compact_log: list[str]    # empty if compaction didn't fire


def run(
    query:     str,
    repo_root: str  = REPO_ROOT,
    strategy:  str  = "auto",   # "auto" | "fuzzy" | "grep" | "semantic"
    top_k:     int  = 8,
) -> RunResult:
    """
    Full retrieval + generation pipeline.

    Parameters
    ----------
    query     : the user's question
    repo_root : path to the repository to search
    strategy  : retrieval strategy; "auto" lets pick_strategy() decide
    top_k     : max candidates to consider before budget_load()
    """
    # ── 1. Manifest ─────────────────────────────────────────────────────────
    manifest     = load_manifest(repo_root)
    manifest_tok = manifest["tokens"]

    # ── 2. Strategy selection ────────────────────────────────────────────────
    strat = pick_strategy(query) if strategy == "auto" else strategy

    # ── 3. Retrieval ─────────────────────────────────────────────────────────
    if strat == "grep":
        candidates = grep_rank(query, repo_root=repo_root)
        # Append fuzzy extras for files with zero grep hits
        found_paths = {f["path"] for f in candidates}
        extras = [f for f in rank_files(glob_files("*.py", repo_root=repo_root), query)
                  if f["path"] not in found_paths]
        candidates = (candidates + extras)[:top_k]

    elif strat == "semantic":
        candidates = semantic_retrieve(query, repo_root=repo_root, top_k=top_k)

    else:   # fuzzy
        candidates = rank_files(glob_files("*.py", repo_root=repo_root), query)[:top_k]

    # ── 4. Budget-aware JIT load ─────────────────────────────────────────────
    loaded = budget_load(candidates, already_used=manifest_tok, repo_root=repo_root)

    # ── 5. Compaction (if needed) ─────────────────────────────────────────────
    hot_tok = sum(f.get("tokens", 0) for f in loaded if f["hot"])
    total   = manifest_tok + hot_tok + count_tokens(query)
    loaded, total, compact_log = compact(loaded, query, total)

    # ── 6. Generate reply ─────────────────────────────────────────────────────
    hot_files = [f for f in loaded if f["hot"]]
    reply, tokens_used = ask_with_manifest(query, repo_root=repo_root, files=hot_files)

    return RunResult(
        reply        = reply,
        strategy     = strat,
        files        = loaded,
        tokens_used  = tokens_used,
        compact_log  = compact_log,
    )

### 10.3 Try all three strategies

Three queries, each routed automatically to a different strategy.
Watch the panel's **Strategy** field change on each run.

In [ ]:
repo = "sample_project"

queries = [
    "Where does the code raise a `TypeError`?",           # → grep
    "Why does the parser wrap scalars in a dict?",        # → semantic
    "What does the formatter do with the delimiter?",     # → fuzzy
]

for q in queries:
    show_rule(f"query: {q}")
    # Before: strategy not yet chosen, no files loaded
    show_panel(
        query       = q,
        token_used  = 0,
        strategy    = "auto",
        prompt_size = count_tokens(q),
    )
    result = run(q, repo_root=repo)
    # After: run() has picked a strategy, loaded files, called the LLM
    show_panel(
        query       = q,
        token_used  = result.tokens_used,
        files       = [{"path": f["path"], "hot": f["hot"]} for f in result.files],
        strategy    = result.strategy,
        prompt_size = count_tokens(q),
    )
    show_reply(result.reply)


---
## Chapter 11 — Write + Diff

**Goal:** build the write primitive — a function you call directly to propose
and apply a file change.

We are not inside an agent loop yet.  You decide what to change and when;
the LLM proposes the new content; you confirm before anything touches disk.
Chapter 12 wires this into the agent loop so the model can make that decision
on its own.

Two principles guide this chapter:

1. **Show before you write.** The diff is always printed before any file is touched.
2. **Full file, not a patch.** The LLM receives the original content and returns
   the complete new file.  We derive the diff from those two strings — no fragile
   line-number arithmetic.

**You will:**
- Write `make_diff()` — unified diff between two strings, rendered as colour HTML
- Write `apply_patch()` — ask the LLM to rewrite a file given a plain-English instruction
- Write `write_file()` — wraps both: show diff, then write on confirmation
- Try it: call `write_file()` directly to add type hints to a function

### 11.1 `make_diff()` — coloured unified diff

Python's `difflib.unified_diff` does the heavy lifting.
`print_diff()` renders the result as colour-coded HTML in the notebook:
green for additions, red for removals, blue for chunk headers.


In [ ]:
import difflib

def make_diff(
    original:  str,
    proposed:  str,
    file_path: str = "<file>",
    context:   int = 3,
) -> str:
    """
    Return a unified diff string between *original* and *proposed*.
    Returns an empty string if they are identical.
    """
    orig_lines = original.splitlines(keepends=True)
    new_lines  = proposed.splitlines(keepends=True)
    diff_lines = list(difflib.unified_diff(
        orig_lines, new_lines,
        fromfile=f"a/{file_path}",
        tofile=f"b/{file_path}",
        n=context,
    ))
    return "".join(diff_lines)


def print_diff(diff: str) -> None:
    """Render a unified diff with colour-coded lines in the notebook."""
    if not diff:
        print("No changes.")
        return
    lines = diff.splitlines()
    html_lines = []
    for ln in lines:
        safe = ln.replace("&","&amp;").replace("<","&lt;")
        if ln.startswith("+") and not ln.startswith("+++"):
            html_lines.append(f'<span style="color:#2da44e">{safe}</span>')
        elif ln.startswith("-") and not ln.startswith("---"):
            html_lines.append(f'<span style="color:#cf222e">{safe}</span>')
        elif ln.startswith("@@"):
            html_lines.append(f'<span style="color:#0969da">{safe}</span>')
        else:
            html_lines.append(f'<span style="color:#57606a">{safe}</span>')
    display(HTML(
        '<pre style="background:#f6f8fa;border:1px solid #d0d7de;border-radius:6px;'
        'padding:12px;font-size:0.82rem;overflow-x:auto;line-height:1.4">'
        + "\n".join(html_lines) + "</pre>"
    ))


In [ ]:
_original = 'def add(a, b):\n    return a + b\n'
_proposed = 'def add(a: int, b: int) -> int:\n    """Return the sum of a and b."""\n    return a + b\n'
print_diff(make_diff(_original, _proposed, "math_utils.py"))


### 11.2 `apply_patch()` — ask the LLM to rewrite a file

We give the model the original file content and a plain-English instruction.
It returns the **complete** new file — not a patch, not just the changed lines.
This is deliberate: asking the model to produce a full file is more reliable
than asking it to produce a diff, which it frequently gets wrong.

In [ ]:
def apply_patch(
    file_path:   str,
    instruction: str,
    repo_root:   str = REPO_ROOT,
) -> tuple[str, str]:
    """
    Ask the LLM to apply *instruction* to the file at *file_path*.

    Returns (original_content, proposed_content).
    The proposed content is the raw LLM output, stripped of markdown fences.
    If the file does not exist yet, original is treated as empty string
    (allows the agent to create new files).
    """
    # Normalise: strip repo_root prefix if the model included it
    rel = file_path
    for prefix in (repo_root + "/", repo_root + os.sep):
        if rel.startswith(prefix):
            rel = rel[len(prefix):]
            break

    full_path = Path(repo_root) / rel
    original  = full_path.read_text(errors="ignore") if full_path.exists() else ""

    prompt = (
        f"Apply the following instruction to the source file below.\n"
        f"Return ONLY the complete modified file — no explanations, "
        f"no markdown fences, no commentary.\n\n"
        f"INSTRUCTION: {instruction}\n\n"
        f"FILE: {rel}\n"
        f"```\n{original}\n```"
    )

    proposed_raw, _ = chat([{"role": "user", "content": prompt}])

    # Strip markdown code fences if the model added them anyway
    proposed = re.sub(r"^```[a-zA-Z]*\n?", "", proposed_raw.strip())
    proposed = re.sub(r"\n?```$", "", proposed)

    return original, proposed.strip() + "\n"

### 11.3 `write_file()` — diff, confirm, write

`write_file()` wraps `apply_patch()`:
- Always shows the diff first
- In `dry_run=True` mode (default) it stops there — nothing is written
- In `dry_run=False` mode it writes the file after showing the diff

In [ ]:
def write_file(
    file_path:   str,
    instruction: str,
    repo_root:   str  = REPO_ROOT,
    dry_run:     bool = True,
    verbose:     bool = True,
) -> tuple[str, str, str]:
    """
    Apply *instruction* to *file_path*, show the diff, optionally write.

    Parameters
    ----------
    file_path   : path relative to repo_root
    instruction : plain-English change request
    repo_root   : repository root
    dry_run     : if True, print the diff but do NOT write to disk
    verbose     : if False, suppress all print output (used when the caller,
                  e.g. _handle_write, handles display itself)

    Returns (original, proposed, diff_text).
    """
    if verbose:
        print(f"\nApplying: {instruction}")
        print(f"File: {file_path}\n")

    original, proposed = apply_patch(file_path, instruction, repo_root)
    diff = make_diff(original, proposed, file_path)

    if verbose:
        show_rule("Proposed diff")
        print_diff(diff)

    if not diff:
        if verbose:
            print("No changes proposed by the model.")
        return original, proposed, diff

    if dry_run:
        if verbose:
            print("\nDRY RUN — file not written. Set dry_run=False to apply.")
    else:
        full_path = Path(repo_root) / file_path
        full_path.write_text(proposed)
        if verbose:
            print(f"\n✓ Written: {full_path}")

    return original, proposed, diff

### 11.4 Try it — add type hints to `_flatten`

Call `write_file()` directly, just like you would from the REPL.
The LLM reads the file, proposes the change, and prints the diff.
Nothing is written until you set `dry_run=False`.

In Chapter 12 the agent loop will make this call automatically as part of a plan.

In [ ]:
original, proposed, diff = write_file(
    file_path   = "utils/parser.py",
    instruction = "Add PEP 484 type annotations to all function signatures. "
                  "Do not change any logic or add any comments.",
    repo_root   = "sample_project",
    dry_run     = True,    # ← change to False to write to disk
)

---
## Chapter 12 — Agent Loop

**Goal:** turn the agent from a question-answerer into an autonomous task executor.

So far each chapter required the user to drive every step.
Chapter 12 adds a loop that:

1. **Plans** — asks the LLM to break the task into ordered steps
2. **Executes** — for each step, decides whether to read (→ `run()`) or write (→ `write_file()`)
3. **Verifies** — after writing, asks the LLM to confirm the change makes sense
4. **Iterates** — repeats until the plan is complete or `max_steps` is reached

The loop is intentionally simple — no tool-calling API, no JSON schema enforcement.
The LLM outputs plain text; the loop parses a lightweight convention:

```
STEP: <description>
ACTION: read | write
TARGET: <file path>   (for write actions)
INSTRUCTION: <what to change>   (for write actions)
```

**You will:**
- Write `plan_task()` — ask the LLM to emit a structured step list
- Write `execute_step()` — dispatch one step to read or write
- Write `agent_loop()` — run the full plan with a step cap and verification

> **⚠️ A note on model quality**
>
> From here on, the notebook asks the LLM to plan, write code, and reason about its own output.
> These are genuinely hard tasks and **model quality matters a lot**.
>
> - `google/gemini-2.5-flash-lite` (the Colab default) is fast and free but is optimised for cost,
>   not code generation. It will sometimes produce correct results and sometimes not — this is
>   expected, not a bug in the notebook.
> - For more reliable results on Chapters 12–15, switch to `google/gemini-2.5-flash` (paid tier,
>   uncomment in the Colab setup cell) or use **Ollama with `qwen3-coder-next:cloud`**, which
>   handles code tasks significantly better.
>
> If a cell produces surprising or wrong output, try running it again — LLMs are non-deterministic.
> If it consistently fails, the model may just not be up to the task. Switch models and retry.

### 12.1 `plan_task()` — break a task into steps

We give the model the manifest and a system prompt that enforces
the `STEP / ACTION / TARGET / INSTRUCTION` format.
The output is parsed into a list of step dicts.

In [ ]:
# ── Action Registry ────────────────────────────────────────────────────────────
#
# Every capability the agent has is an "action".  An action bundles four things:
#
#   description  one sentence explaining what it does (injected into _PLAN_SYSTEM)
#   fields       which STEP: fields this action uses — TARGET, CMD, URL, MESSAGE, …
#   example      a complete STEP: block the LLM can imitate
#   handler      function(step, repo_root) → dict that actually runs the step
#   rule         optional constraint added to the Rules section of _PLAN_SYSTEM
#
# ─── Why this matters ─────────────────────────────────────────────────────────
# Without a registry, adding a new capability (like ACTION: fetch in Chapter 14)
# meant three separate edits: update _PLAN_SYSTEM, update plan_task, update
# execute_step.  Miss one and the agent silently breaks.
#
# With the registry, adding a capability is one call:
#   register_action("fetch", ..., handler=_handle_fetch)
#   _PLAN_SYSTEM = build_plan_system()
#
# plan_task() and execute_step() never need to change.

ACTION_REGISTRY: dict[str, dict] = {}


def register_action(
    name:        str,
    description: str,
    fields:      list[str],
    example:     str,
    handler,                    # Callable[[dict, str], dict]
    rule:        str = "",
) -> None:
    """
    Register a new action.

    name        — lowercase key the LLM writes after "ACTION:", e.g. "fetch"
    description — one sentence that describes the action to the LLM
    fields      — STEP: fields this action uses, e.g. ["URL"]
    example     — a full STEP: block the LLM can copy
    handler     — function(step, repo_root) -> dict, called by execute_step()
    rule        — optional rule appended to the Rules section of _PLAN_SYSTEM
    """
    ACTION_REGISTRY[name] = {
        "description": description,
        "fields":      [f.upper() for f in fields],
        "example":     example,
        "handler":     handler,
        "rule":        rule,
    }


def build_plan_system() -> str:
    """
    Assemble _PLAN_SYSTEM from the current ACTION_REGISTRY.

    Call this after every register_action() to keep the LLM prompt in sync.
    The output is a plain string — the same type as the hand-written version
    it replaces, so the rest of the notebook works unchanged.
    """
    # One example block per registered action, separated by blank lines
    action_blocks = "\n\n".join(a["example"] for a in ACTION_REGISTRY.values())

    # Per-action rules (omit empty strings), then universal rules
    action_rules = [a["rule"] for a in ACTION_REGISTRY.values() if a["rule"]]
    common_rules = [
        "Never invent file paths; never use absolute or /tmp paths.",
        "READ returns raw file contents — it does NOT transform or improve the code. "
        "Always follow a READ step with the WRITE steps that apply the actual changes.",
        "Your response must be the COMPLETE plan — every step from start to finish. "
        "Do not stop after a READ or FETCH step.",
        "Never place a READ step immediately after a FETCH step. "
        "FETCH already puts the content in the thread — a READ cannot re-read a URL.",
        "Never use a URL as a READ TARGET. READ only accepts file paths or glob patterns.",
        "Each WRITE step targets exactly one file.",
        "Be specific in INSTRUCTION — the change will be applied by another model "
        "with no extra context.",
        "Do not add any text outside the step blocks.",
    ]
    rules = "\n".join(f"- {r}" for r in action_rules + common_rules)

    plan = (
        "You are a coding agent. Output the COMPLETE ordered plan to fully finish the task.\n"
        "Include every step — do not stop after gathering information.\n\n"
        "Each step must be one of these formats (blank line between steps):\n\n"
        f"{action_blocks}\n\n"
        f"Rules:\n{rules}\n"
    )

    return plan


In [ ]:
# ── Chapter 12 action handlers ─────────────────────────────────────────────────
#
# Each handler is a plain function: (step, repo_root) → dict.
# The dict always has a "type" field matching the action name.
# execute_step() (defined in 12.2) looks up the right handler automatically.


def _handle_read(step: dict, repo_root: str) -> dict:
    """
    Read source files and return their raw content — no LLM call.

    TARGET should be a file path or glob pattern, e.g.:
      "utils/parser.py"   — single file
      "utils/*.py"        — all .py files in utils/
      "utils/"            — same (trailing slash expanded to utils/*.py)

    The raw content is returned so subsequent WRITE steps can reference
    the current code without the agent having to guess what's already there.
    """
    target = step.get("target", "").strip()

    # Try target directly as a glob pattern (handles "utils/*.py", "**/*.py", etc.)
    files = tool_glob(target, repo_root=repo_root)

    # "utils/" or "utils" → expand to "utils/*.py"
    if not files:
        clean = target.rstrip("/")
        files = tool_glob(f"{clean}/*.py", repo_root=repo_root)

    # Direct file path fallback
    if not files:
        candidate = Path(repo_root) / target
        if candidate.is_file():
            try:
                files = [{"path": str(candidate.relative_to(repo_root)),
                          "bytes": candidate.stat().st_size, "hot": False}]
            except OSError:
                pass

    if not files:
        return {"type": "read", "output": f"No files found matching: {target!r}", "ok": False}

    parts = []
    for f in files[:8]:   # cap at 8 files to avoid flooding the context
        loaded = jit_read(f, repo_root=repo_root)
        parts.append(f"--- FILE: {loaded['path']} ({loaded['tokens']} tokens) ---\n{loaded['content']}")

    return {
        "type":   "read",
        "output": "\n\n".join(parts),
        "files":  [f["path"] for f in files[:8]],
        "ok":     True,
    }


def _handle_write(step: dict, repo_root: str) -> dict:
    """Generate a file diff and return it for the user to review and accept."""
    original, proposed, diff = write_file(
        file_path   = step["target"],
        instruction = step["instruction"],
        repo_root   = repo_root,
        dry_run     = True,    # always dry-run here; _display_result writes on accept
        verbose     = False,   # _display_result handles all display via show_details
    )
    return {"type": "write", "file_path": step["target"],
            "new_content": proposed, "diff": diff}


def _handle_bash(step: dict, repo_root: str) -> dict:
    """Run a shell command and capture stdout + stderr."""
    cmd  = step.get("cmd", "")
    proc = tool_bash(cmd, repo_root=repo_root)
    return {"type": "bash", "cmd": cmd,
            "output": (proc["stdout"] + proc["stderr"]).strip(),
            "ok": proc["code"] == 0}


def _handle_glob(step: dict, repo_root: str) -> dict:
    """Return file paths that match a glob pattern."""
    pattern = step.get("pattern", "**/*")
    files   = tool_glob(pattern, repo_root=repo_root)
    return {"type": "glob", "pattern": pattern,
            "files": [f["path"] for f in files]}


def _handle_grep(step: dict, repo_root: str) -> dict:
    """Search file contents for a regex and return matching paths."""
    pattern = step.get("pattern", "")
    matches = tool_grep(pattern, repo_root=repo_root)
    return {"type": "grep", "pattern": pattern,
            "matches": [f["path"] for f in matches]}


# ── Register the Chapter 12 actions ───────────────────────────────────────────

register_action(
    name        = "read",
    description = "Read the current content of one or more source files — no LLM analysis.",
    fields      = ["TARGET"],
    example     = (
        "STEP: Read all Python files in utils/ to see current function signatures\n"
        "ACTION: read\n"
        "TARGET: utils/*.py"
    ),
    handler     = _handle_read,
    rule        = (
        "Use ACTION: read to get the current file contents before writing. "
        "TARGET must be a file path or glob pattern (e.g. utils/*.py), not a question."
    ),
)

register_action(
    name        = "write",
    description = "Generate or edit a file — the LLM produces the full content.",
    fields      = ["TARGET", "INSTRUCTION"],
    example     = (
        "STEP: Add type hints to all functions in utils/parser.py\n"
        "ACTION: write\n"
        "TARGET: utils/parser.py\n"
        "INSTRUCTION: Add PEP-484 type annotations to every function signature."
    ),
    handler     = _handle_write,
)

register_action(
    name        = "bash",
    description = "Run a shell command (copy, move, compile, run tests, etc.).",
    fields      = ["CMD"],
    example     = (
        "STEP: Copy the template to the docs folder\n"
        "ACTION: bash\n"
        "CMD: cp templates/readme.md docs/README.md"
    ),
    handler     = _handle_bash,
    rule        = "Prefer ACTION: bash for file system operations (copy, move, delete, rename).",
)

register_action(
    name        = "glob",
    description = "List files whose path matches a glob pattern. Supports ** for recursion.",
    fields      = ["PATTERN"],
    example     = (
        "STEP: List all Python files under utils/\n"
        "ACTION: glob\n"
        "PATTERN: utils/*.py"
    ),
    handler     = _handle_glob,
    rule        = "Use ACTION: glob to find files by name pattern. Prefer utils/*.py over utils/**/*.py for a flat directory.",
)

register_action(
    name        = "grep",
    description = "Search file contents for a regex pattern.",
    fields      = ["PATTERN"],
    example     = (
        "STEP: Find all files that raise TypeError\n"
        "ACTION: grep\n"
        "PATTERN: raise TypeError"
    ),
    handler     = _handle_grep,
    rule        = "Use ACTION: grep to search file contents for a regex pattern.",
)

# Build the system prompt from everything registered so far.
_PLAN_SYSTEM = build_plan_system()

In [ ]:
def plan_task(task: str, repo_root: str = REPO_ROOT) -> list[dict]:
    """
    Ask the LLM to produce a step-by-step plan for *task*.

    The field regex and step-dict keys are derived automatically from
    ACTION_REGISTRY — no manual update needed when new actions are added.

    Returns a list of step dicts with at minimum:
        {"step": str, "action": str, <action-specific fields>: str, ...}
    """
    manifest  = load_manifest(repo_root)
    all_files = glob_files("*", repo_root=repo_root)
    file_list = "\n".join(f["path"] for f in all_files)

    prompt = (
        f"PROJECT MAP:\n{manifest['text']}\n\n"
        f"AVAILABLE FILES (use only if the task involves the codebase):\n{file_list}\n\n"
        f"TASK: {task}"
    )

    raw, _ = chat([
        {"role": "system", "content": _PLAN_SYSTEM},
        {"role": "user",   "content": prompt},
    ])

    # Build the field regex from whatever is currently in the registry.
    # STEP and ACTION are always present; action-specific fields come from
    # each action's "fields" list.  When register_action() is called with a
    # new field (e.g. URL for fetch) it appears here
    # automatically — no edit to plan_task required.
    known_fields = {"STEP", "ACTION"} | {
        f for a in ACTION_REGISTRY.values() for f in a["fields"]
    }
    field_pat = "|".join(sorted(known_fields))

    # All lowercase field names we may need in the returned step dicts
    all_field_names = {f.lower() for a in ACTION_REGISTRY.values() for f in a["fields"]}

    steps = []
    for block in re.split(r"(?m)(?=^STEP:)", raw.strip()):
        if not block.strip():
            continue
        parsed = {
            m.group(1).lower(): m.group(2).strip()
            for line in block.splitlines()
            if (m := re.match(rf"^({field_pat}):\s*(.+)$", line, re.IGNORECASE))
        }
        action = parsed.get("action", "").lower()
        # A step is valid only if its action is registered — the registry is the
        # single source of truth for what actions exist.
        if action in ACTION_REGISTRY:
            steps.append({
                "step":   parsed.get("step", ""),
                "action": action,
                **{f: parsed.get(f, "") for f in all_field_names},
            })

    # ── Post-planning sanity filter ───────────────────────────────────────────
    # Remove READ steps that immediately follow a FETCH step.
    # The model sometimes tries to "read back" fetched content, treating
    # the URL as a file path — but FETCH already put the content in the
    # thread.  This structural check is more reliable than a prompt rule.
    filtered    = []
    prev_action = None
    for s in steps:
        spurious_read = (
            s["action"] == "read"
            and prev_action == "fetch"
        )
        if not spurious_read:
            filtered.append(s)
        prev_action = s["action"]
    return filtered


### 12.2 Tools and `execute_step()`

Three tools give the agent the ability to interact with the environment directly,
rather than just reading and proposing changes:

| Tool | What it does |
|------|-------------|
| `bash` | Run any shell command — returns stdout, stderr, and exit code |
| `glob` | Find files by pattern — fast, no file content read |
| `grep` | Search file contents by regex — returns matching files with snippets |

`execute_step()` is the dispatcher: it routes each plan step to the right handler —
`run()` for reads, `write_file()` for writes, and `call_tool()` for tool steps.


In [ ]:
CURRENT_REPO_ROOT = REPO_ROOT


def _resolve_root(repo_root: str | None = None) -> str:
    """Return repo_root if given, otherwise fall back to CURRENT_REPO_ROOT."""
    return repo_root or CURRENT_REPO_ROOT


def tool_bash(cmd: str, repo_root: str | None = None) -> dict:
    """
    Run a shell command in repo_root. Returns stdout, stderr, and exit code.

    shell=True lets the command string use shell features: pipes (|), redirects (>),
    glob expansion (*.py), and chained commands (&&).  The trade-off is that the
    command string must never be assembled from untrusted user input — if it were,
    a user could inject arbitrary shell commands.  Inside an agent, commands come
    from the LLM's plan, which is already constrained by _PLAN_SYSTEM.
    """
    root   = _resolve_root(repo_root)
    result = subprocess.run(cmd, shell=True, cwd=root, capture_output=True, text=True)
    return {
        "cmd":    cmd,
        "cwd":    root,
        "code":   result.returncode,
        "stdout": result.stdout,
        "stderr": result.stderr,
    }


def tool_glob(pattern: str, repo_root: str | None = None) -> list[dict]:
    """
    Find files matching a glob pattern (e.g. 'utils/**/*.py').

    Uses Path.glob() so '**' recursive patterns work correctly.
    Skips hidden directories and common noise dirs (same as glob_files).
    Returns file metadata dicts — no content loaded.
    """
    root    = Path(_resolve_root(repo_root))
    matches = []
    for full in root.glob(pattern):
        if not full.is_file():
            continue
        # Skip hidden dirs and noise dirs in any path component
        if any(p.startswith(".") or p in SKIP_DIRS
               for p in full.relative_to(root).parts[:-1]):
            continue
        try:
            size = full.stat().st_size
        except OSError:
            continue
        matches.append({
            "path":  str(full.relative_to(root)),
            "bytes": size,
            "hot":   False,
        })
    return sorted(matches, key=lambda x: x["path"])


def tool_grep(pattern: str, repo_root: str | None = None) -> list[dict]:
    """Search file contents for a regex pattern. Returns matching files with line snippets."""
    return grep_repo(pattern, repo_root=_resolve_root(repo_root))

In [ ]:
def execute_step(step: dict, repo_root: str = REPO_ROOT) -> dict:
    """
    Execute one plan step by dispatching to the registered action handler.

    The handler is looked up in ACTION_REGISTRY by step['action'].  Any action
    added via register_action() is automatically available here — this function
    never needs to change when new actions are added.

    Return value shape (depends on the handler):

      read   → {"type": "read",   "output": str}
      write  → {"type": "write",  "file_path": str, "new_content": str, "diff": str}
      bash   → {"type": "bash",   "cmd": str, "output": str, "ok": bool}
      glob   → {"type": "glob",   "pattern": str, "files": list[str]}
      grep   → {"type": "grep",   "pattern": str, "matches": list[str]}
      fetch  → {"type": "fetch",  "url": str, "output": str, "ok": bool}   # Chapter 14+
    """
    action = step.get("action", "read")
    if action not in ACTION_REGISTRY:
        registered = list(ACTION_REGISTRY)
        return {"type": "error",
                "output": f"Unknown action {action!r}. Registered: {registered}"}
    return ACTION_REGISTRY[action]["handler"](step, repo_root)


### 12.3 `agent_loop()` — plan → execute → verify

The loop runs the full plan, printing progress at each step.
After all write steps complete, it runs a final verification query
to ask the model whether the changes look correct.

In [ ]:
def agent_loop(
    task:      str,
    repo_root: str  = REPO_ROOT,
    dry_run:   bool = True,
    max_steps: int  = 8,
) -> list[dict]:
    """
    Autonomous task execution loop.

    1. Plan the task with plan_task()
    2. Execute each step with execute_step()
       - WRITE steps always show a diff but never touch disk
         (_handle_write passes dry_run=True to write_file internally)
       - Set dry_run=False to allow the Ch14 agent_loop to accept writes
    3. Verify written files exist on disk (Ch14 loop only)

    Returns the step log (list of dicts with "step", "action", "status").
    """
    show_rule(f"Agent Loop  ·  {task[:60]}")
    print(f"repo: {repo_root}  |  dry_run={dry_run}  |  max_steps={max_steps}\n")

    # ── 1. Plan ───────────────────────────────────────────────────────────────
    plan = plan_task(task, repo_root=repo_root)
    if not plan:
        print("Planning failed — no steps parsed from model output.")
        return []

    print(f"Plan: {len(plan)} step(s)\n")

    # ── 2. Execute ────────────────────────────────────────────────────────────
    log = []
    for i, step in enumerate(plan[:max_steps], 1):
        show_rule(f"Step {i}/{min(len(plan), max_steps)}  {step['action'].upper()}  {step['step']}")

        # All steps — including write — go through execute_step.
        # _handle_write always calls write_file(dry_run=True) so no file is
        # ever written here; the diff is printed inline by write_file itself.
        status = execute_step(step, repo_root=repo_root)

        if status.get("type") == "write":
            # write_file already printed the diff above; just note the target
            print(f"\n↳ {'DRY RUN' if dry_run else 'WRITTEN'}  {status.get('file_path', step.get('target', ''))}")
        else:
            print(f"↳ {status}")

        log.append({**step, "status": status})

    # ── 3. Verify ─────────────────────────────────────────────────────────────
    written = [s for s in log if s["action"] == "write"]
    if written and not dry_run:
        show_rule("Verification")
        all_ok = True
        for s in written:
            path = Path(repo_root) / s["target"]
            icon = "✅" if path.exists() else "❌"
            print(f"{icon} {s['target']}")
            if not path.exists():
                all_ok = False
        print("\nAll files confirmed on disk." if all_ok else "\nSome files are missing.")
    elif written and dry_run:
        print("\nDRY RUN — files were previewed above but not written to disk.")

    show_rule("Loop complete")
    return log

### 12.4 Try it

Run the agent loop on a real task.
`dry_run=True` means no files are touched — you can set it to `False`
once you've reviewed the diffs and are happy with the plan.

### 12.5 Tool demo — syntax check

`tool_bash` lets the agent act on the repo, not just read it.
Here we call it directly to check every Python file for syntax errors — the same
call the agent emits when its plan includes:

```
ACTION: tool
TOOL: bash
ARGS: {"cmd": "find . -name '*.py' -exec python -m py_compile {} +"}
```

`py_compile` is Python's built-in syntax checker. It exits with code 0 if every
file parses cleanly, or prints the offending line and exits with code 1.


In [ ]:
import sys as _sys

result = tool_bash(
    f"{_sys.executable} -m py_compile $(find . -name '*.py') 2>&1 && echo 'All OK'",
    repo_root="sample_project",
)

if result["code"] == 0:
    print("✓ All Python files pass syntax check.")
else:
    print("Syntax errors found:\n")
    print(result["stdout"] or result["stderr"])


In [ ]:
log = agent_loop(
    task      = "Add type annotations to all functions in utils/ and update their docstrings.",
    repo_root = "sample_project",
    dry_run   = True,    # ← change to False to write files
    max_steps = 6,
)

---
## Chapter 13 — Test Generation

**Goal:** the agent writes its own tests, runs them, and fixes failures.

This is the most complete demonstration of the full system.
Every function built across chapters 1–11 is in play:

- `run()` reads the source file to understand it (Ch 9)
- `write_file()` writes the test file to disk (Ch 10)
- `agent_loop()` orchestrates the plan (Ch 11)
- A subprocess call runs `pytest` and captures the output
- If tests fail, the failure output is fed back to `apply_patch()` and retried

**You will:**
- Write `generate_tests()` — ask the LLM to produce a pytest file for a module
- Write `run_tests()` — execute pytest, capture pass/fail/error
- Write `test_loop()` — generate → run → fix loop with a retry cap
- Try it: generate tests for `utils/formatter.py` and run them

### 13.1 `generate_tests()` — produce a pytest file

We read the source module first via `run()`, then ask the model
to write a complete `pytest` test file for it.
The test file is returned as a string — not written yet.

In [ ]:
def generate_tests(
    source_path: str,
    repo_root:   str = REPO_ROOT,
) -> str:
    """
    Generate a pytest test file for the module at *source_path*.

    Returns the test file content as a string (not written to disk yet).
    """
    full_path = Path(repo_root) / source_path
    source    = full_path.read_text(errors="ignore")

    prompt = (
        f"Write a complete pytest test file with a test class for the following Python module.\n\n"
        f"Requirements:\n"
        f"- Use pytest (not unittest)\n"
        f"- Cover the main happy path and at least two edge cases per function\n"
        f"- Import from the module using its dotted path relative to the repo root\n"
        f"  (e.g. 'from utils.formatter import to_csv')\n"
        f"- Return ONLY the test file — no explanation, no markdown fences\n\n"
        f"SOURCE FILE: {source_path}\n\n{source}"
    )

    raw, _ = chat([{"role": "user", "content": prompt}])

    # Strip fences if model added them
    code = re.sub(r"^```[a-zA-Z]*\n?", "", raw.strip())
    code = re.sub(r"\n?```$", "", code)
    return code.strip() + "\n"


In [ ]:
show_rule("Generating tests for utils/formatter.py")
test_code = generate_tests("utils/formatter.py", repo_root="sample_project")
display(Markdown(f"```python\n{test_code}\n```"))

### 13.2 `run_tests()` — execute pytest, capture results

We run `pytest` in a subprocess and return a structured result:
passed, failed, error count, and the raw output for feeding back
to the model if something went wrong.

In [ ]:
import subprocess as _sp


def run_tests(
    test_path: str,
    repo_root: str = REPO_ROOT,
) -> dict:
    """
    Run pytest on *test_path* (relative to repo_root).

    Returns:
        {
          "passed":  int,
          "failed":  int,
          "errors":  int,
          "output":  str,   # full pytest stdout+stderr
          "ok":      bool,  # True if passed > 0 and failed == errors == 0
        }
    """
    result = _sp.run(
        [sys.executable, "-m", "pytest", test_path, "-v", "--tb=short",
         "--no-header", "-q"],
        capture_output=True,
        text=True,
        cwd=repo_root,   # run from inside repo_root, so test_path is relative
    )

    output = result.stdout + result.stderr

    # Parse summary line: "3 passed, 1 failed, 0 errors"
    passed = int(m.group(1)) if (m := re.search(r"(\d+) passed",  output)) else 0
    failed = int(m.group(1)) if (m := re.search(r"(\d+) failed",  output)) else 0
    errors = int(m.group(1)) if (m := re.search(r"(\d+) error",   output)) else 0

    return {
        "passed": passed,
        "failed": failed,
        "errors": errors,
        "output": output,
        "ok":     passed > 0 and failed == 0 and errors == 0,
    }

### 13.3 `test_loop()` — generate → run → fix → retry

The loop:
1. Generate a test file with `generate_tests()`
2. Run it with `run_tests()`
3. If tests fail, build a fix instruction from the **live pytest output** and rewrite the file
4. Repeat until tests pass or retries are exhausted

**Why not just use `agent_loop`?**

`agent_loop` plans upfront — it decides all steps before executing any of them. That works well when the task is predictable ("add type annotations to these three files"). It breaks down here because:

- The number of iterations is dynamic — we don't know in advance whether the generated tests will pass on the first try.
- Step 3's fix instruction depends on the *actual output* of step 2. That output doesn't exist at plan time.

`agent_loop` has no way to feed a step's result back into a later step's instruction. `test_loop` is a **feedback loop**, not a plan-then-execute sequence.

**What is reflection?**

A *reflecting* agent observes the result of each step before deciding the next one. After running pytest and seeing failures, it asks: *"Given what just happened, what should I do next?"* rather than following a fixed plan. This is sometimes called a ReAct loop (Reason + Act).

**Combining the strategies**

A unified loop would work like this:

```
plan → execute step → observe result → re-plan → execute step → ...
```

After each step, the agent appends the result to its context and asks the planner whether the remaining steps still make sense, or whether new steps are needed. `test_loop` is a hand-written specialisation of this pattern for the generate/test/fix cycle.

We will build this generalised reflection loop in a later chapter, at which point `test_loop` becomes a one-line call to `agent_loop` with reflection enabled.


In [ ]:
def test_loop(
    source_path: str,
    repo_root:   str = REPO_ROOT,
    max_retries: int = 3,
) -> dict:
    """
    Full generate → run → fix loop for *source_path*.

    Returns the final run_tests() result dict plus a "attempts" key.
    """
    # Derive test file path: utils/formatter.py → tests/test_formatter_gen.py
    stem      = Path(source_path).stem
    test_path = f"tests/test_{stem}_gen.py"
    full_test = Path(repo_root) / test_path
    full_test.parent.mkdir(parents=True, exist_ok=True)

    show_rule(f"Test Loop  ·  {source_path}")

    # ── 1. Generate initial tests ──────────────────────────────────────────────
    print("Step 1: generating tests…")
    test_code = generate_tests(source_path, repo_root=repo_root)
    full_test.write_text(test_code)
    print(f"Written: {test_path}")

    for attempt in range(1, max_retries + 2):   # +1 for initial attempt
        # ── 2. Run tests ──────────────────────────────────────────────────────
        show_rule(f"Attempt {attempt}")
        result = run_tests(test_path, repo_root=repo_root)

        color = "green" if result["ok"] else "red"
        icon = "✓" if result["ok"] else "✗"
        print(f"  {icon} passed={result['passed']}  failed={result['failed']}  errors={result['errors']}")

        if result["ok"]:
            print("\n✓ All tests pass.")
            result["attempts"] = attempt
            return result

        if attempt > max_retries:
            print(f"\nMax retries ({max_retries}) reached. Giving up.")
            break

        # ── 3. Fix ────────────────────────────────────────────────────────────
        print("\nTests failed. Asking model to fix…")
        print(result['output'][-1200:])   # last 1200 chars of pytest output

        fix_instruction = (
            f"The test file has failures. Fix ONLY the test code — do not modify the "
            f"source module. Here is the pytest output:\n\n{result['output'][-1500:]}"
        )
        _, fixed_code = apply_patch(test_path, fix_instruction, repo_root=repo_root)
        full_test.write_text(fixed_code)
        print(f"Rewritten: {test_path}")

    result["attempts"] = max_retries + 1
    return result

### 13.4 Try it — generate and run tests for `formatter.py`

This cell writes real files to `sample_project/tests/` and runs `pytest`.
Make sure `pytest` is installed (`pip install pytest`).

In [ ]:
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "pytest"],
    stdout=subprocess.DEVNULL,
)

result = test_loop(
    source_path = "utils/formatter.py",
    repo_root   = "sample_project",
    max_retries = 10, # <- increase this if it continues to fail
)

icon = "✓ PASS" if result["ok"] else "✗ FAIL"
print(f"\nFinal result: {icon}  in {result['attempts']} attempt(s)")

---

## Chapter 14 — Adding a New Capability

The agent is a pipeline of functions connected by a small number of **switch points** — places where the code checks an action type or strategy name and branches. Adding a new capability always means the same three things:

1. **Write the function** — pure Python, no magic.
2. **Wire it in** — add a case to the right switch point.
3. **Tell the planner** — add a line to `_PLAN_SYSTEM` so the LLM knows the action exists.

That's the whole pattern. Everything downstream — the agent loop, the Streamlit UI, the activity log — picks it up automatically.

### The three switch points

| What you're adding | Switch point |
|---|---|
| New retrieval strategy | `pick_strategy()` + new `*_retrieve()` function |
| New plan action | `_PLAN_SYSTEM` prompt + `plan_task()` parser + `execute_step()` |
| New file types to index/embed | `CODE_EXTENSIONS` set |

### 14.1 The worked example: `ACTION: fetch`

We'll add the ability for the agent to fetch a URL — useful when the task says "implement this API" and you paste in a link to the docs.

The plan action will look like:

```
STEP: Read the requests library docs
ACTION: fetch
URL: https://docs.python-requests.org/en/latest/
```

Three changes required. We'll do them one at a time.


### 14.1 — Write the function

`fetch_url()` downloads a page and strips HTML tags so the model gets readable text rather than a wall of `<div>` soup. Nothing agent-specific here — just plain Python.


In [ ]:
import re as _re
import requests

def fetch_url(url: str) -> dict:
    """
    Fetch *url* and return plain text (HTML tags stripped).

    Returns {"url": str, "text": str, "ok": bool, "error": str | None}
    """
    try:
        resp = requests.get(url, timeout=15, headers={"User-Agent": "pocket-agent/1.0"})
        resp.raise_for_status()
        # Strip every HTML tag with a simple regex — good enough for docs pages
        text = _re.sub(r"<[^>]+>", " ", resp.text)
        # Collapse whitespace
        text = _re.sub(r"\s{2,}", "\n", text).strip()
        return {"url": url, "text": text, "ok": True, "error": None}
    except Exception as exc:
        return {"url": url, "text": "", "ok": False, "error": str(exc)}


In [ ]:
result = fetch_url("https://httpbin.org/html")
print(f"ok={result['ok']}  chars={len(result['text'])}")
print(result["text"][:300])


### 14.2 — Register the action

With the action registry in place, adding `fetch` is three steps:

1. Write `_handle_fetch()` — a thin wrapper that calls `fetch_url()` and returns a structured dict.
2. Call `register_action("fetch", ...)` to plug it into the registry.
3. Call `_PLAN_SYSTEM = build_plan_system()` to refresh the LLM prompt.

`plan_task()` and `execute_step()` do **not** change — they read from the registry automatically.


In [ ]:
# ── Chapter 14: add ACTION: fetch ─────────────────────────────────────────────

FETCH_DISPLAY_CHARS = 300    # characters shown in the notebook preview only
                             # the full content is always passed to subsequent steps

def _handle_fetch(step: dict, repo_root: str) -> dict:
    """
    Fetch a URL and return its full text content — no LLM call.

    Returns {"type": "fetch", "url": str, "output": str, "ok": bool}.
    The complete text is stored in the thread so subsequent WRITE steps
    have all the material they need to summarise or transform.
    """
    url    = step.get("url", step.get("target", ""))
    result = fetch_url(url)
    if not result["ok"]:
        return {"type": "fetch", "url": url,
                "output": f"Error: {result['error']}", "ok": False}
    return {"type": "fetch", "url": url, "output": result["text"], "ok": True}


register_action(
    name        = "fetch",
    description = "Download an external URL and return its raw text content.",
    fields      = ["URL"],
    example     = (
        "STEP: Download the Python pathlib documentation\n"
        "ACTION: fetch\n"
        "URL: https://docs.python.org/3/library/pathlib.html"
    ),
    handler     = _handle_fetch,
    rule = (
        "Use ACTION: fetch — never ACTION: bash with curl or wget — "
        "when the task references an external URL or documentation link. "
        "After FETCH, the content is automatically available to subsequent WRITE steps "
        "via the thread — never add a READ step to re-read fetched content."
    ),
)

# Rebuild the prompt so the planner knows about fetch.
_PLAN_SYSTEM = build_plan_system()

### 14.x — Adding `readxml`

RSS feeds and XML APIs return tag-heavy markup that models struggle to summarise — the signal (article titles, descriptions) is buried in angle brackets.

`readxml` is a one-step fix: fetch the URL **and** strip the XML in a single action, handing the model clean readable text. The implementation is ~15 lines of stdlib `xml.etree.ElementTree` — no new dependencies.

Notice that adding it requires only:
1. A new handler function `_handle_readxml()`
2. One `register_action("readxml", …)` call

`plan_task()` and `execute_step()` are untouched — the registry pattern paying off again.

In [ ]:
import xml.etree.ElementTree as ET
import re as _re

def _xml_to_text(xml_str: str) -> str:
    """
    Extract readable text from XML / RSS, stripping all markup.

    Strategy:
      1. Parse with ElementTree and pull out the text content of every
         element whose tag is one of the common RSS/Atom signal tags
         (title, description, summary, content, pubDate).
      2. Fall back to a regex tag-strip if the XML is malformed.

    Returns a plain-text string, one "TAG: value" line per element found.
    """
    signal_tags = {"title", "description", "summary", "content", "pubdate", "link"}
    try:
        root  = ET.fromstring(xml_str)
        lines = []
        for elem in root.iter():
            # Strip XML namespace prefix, e.g. "{http://...}title" → "title"
            tag = elem.tag.split("}")[-1].lower()
            if tag in signal_tags and elem.text and elem.text.strip():
                lines.append(f"{tag.upper()}: {elem.text.strip()}")
        if lines:
            return "\n".join(lines)
    except ET.ParseError:
        pass
    # Fallback: regex strip — handles malformed / HTML-ish responses
    text = _re.sub(r"<[^>]+>", " ", xml_str)
    return _re.sub(r" {2,}", " ", text).strip()


def _handle_readxml(step: dict, repo_root: str) -> dict:
    """
    Fetch a URL and return its content with XML/RSS tags stripped.

    Identical to FETCH but passes the raw text through _xml_to_text()
    before storing it in the thread.  Use this instead of FETCH when
    the URL returns XML or an RSS feed — the model receives clean
    title/description lines rather than a wall of markup tags.
    """
    url    = step.get("url", step.get("target", ""))
    result = fetch_url(url)
    if not result["ok"]:
        return {"type": "fetch", "url": url,
                "output": f"Error: {result['error']}", "ok": False}

    clean = _xml_to_text(result["text"])
    return {"type": "fetch", "url": url, "output": clean, "ok": True}


register_action(
    name        = "readxml",
    description = "Fetch a URL and return its text with XML/RSS tags stripped — use for RSS feeds and XML APIs instead of plain fetch.",
    fields      = ["URL"],
    example     = (
        "STEP: Fetch and parse the Slashdot RSS feed\n"
        "ACTION: readxml\n"
        "URL: https://rss.slashdot.org/Slashdot/slashdotMain"
    ),
    handler     = _handle_readxml,
    rule        = "Use ACTION: readxml instead of ACTION: fetch when the URL is an RSS feed or returns XML.",
)

_PLAN_SYSTEM = build_plan_system()
print("readxml registered.")

In [ ]:
# Chapter 14 redefinition of agent_loop — changes from Chapter 12:
#   1. Handles structured dict results from execute_step()
#   2. Shows all output as collapsible details blocks
#   3. For write steps: auto-writes by default; set confirm_write=True to review each diff
#   4. Verifies written files exist on disk

def agent_loop(
    task:          str,
    repo_root:     str  = REPO_ROOT,
    confirm_write: bool = False,
) -> list[dict]:
    """
    Autonomous task execution loop.

    1. Plan the task with plan_task()
    2. Execute each step with execute_step() — returns structured dicts
    3. For write steps:
         confirm_write=False (default) — write immediately, show diff collapsed
         confirm_write=True            — show diff expanded, ask before writing
    4. Verify all accepted writes exist on disk

    Returns the step log (list of dicts with "step", "action", "result", "accepted").
    """
    show_rule(f"Agent Loop  ·  {task[:60]}")
    display(HTML(f'<span style="font-size:0.82rem;color:#57606a">repo: {repo_root}</span>'))

    # ── 1. Plan ───────────────────────────────────────────────────────────────
    plan = plan_task(task, repo_root=repo_root)
    if not plan:
        print("Planning failed — no steps parsed from model output.")
        return []

    display(HTML(f'<span style="font-size:0.82rem;color:#57606a">Plan: {len(plan)} step(s)</span>'))

    # ── 2. Execute ────────────────────────────────────────────────────────────
    log = []
    for i, step in enumerate(plan, 1):
        show_rule(f"Step {i}/{len(plan)}  {step['action'].upper()}  {step['step']}")
        result   = execute_step(step, repo_root=repo_root)
        accepted = False

        if result["type"] == "write":
            diff      = result.get("diff", "")
            file_path = result["file_path"]

            if not diff:
                show_details(
                    f"⚠️  WRITE  {file_path}  — no changes proposed",
                    "The model returned the same content as the original file.\n"
                    "The instruction or context may need to be more specific.",
                    open=True,
                )
            elif confirm_write:
                # Review mode: expand the diff so the user can read it, then prompt.
                show_details(f"✏️  WRITE  {file_path}", diff, open=True)
                if input("\nAccept this write? [y/N] ").strip().lower() == "y":
                    path = Path(repo_root) / file_path
                    path.parent.mkdir(parents=True, exist_ok=True)
                    path.write_text(result["new_content"])
                    display(HTML('<span style="font-size:0.82rem;color:#2da44e">✅ Written</span>'))
                    accepted = True
                else:
                    display(HTML('<span style="font-size:0.82rem;color:#57606a">⏭  Skipped</span>'))
            else:
                # Auto mode (default): write immediately, show diff collapsed.
                path = Path(repo_root) / file_path
                path.parent.mkdir(parents=True, exist_ok=True)
                path.write_text(result["new_content"])
                accepted = True
                show_details(f"✏️  WRITE  {file_path}  ✅ written", diff, open=False)

        elif result["type"] == "fetch":
            ok      = result.get("ok", True)
            icon    = "✅" if ok else "❌"
            preview = result["output"][:FETCH_DISPLAY_CHARS]
            if len(result["output"]) > FETCH_DISPLAY_CHARS:
                preview += f"\n\n[… {len(result['output']) - FETCH_DISPLAY_CHARS} more chars not shown]"
            show_details(f"{icon}  FETCH  {result['url']}", preview, open=not ok)

        elif result["type"] == "bash":
            ok   = result.get("ok", True)
            icon = "✅" if ok else "❌"
            out  = result.get("output", "")
            show_details(f"{icon}  BASH  `{result['cmd']}`", out[:2000], open=not ok)

        elif result["type"] == "read":
            files = result.get("files", [])
            label = (f"📖  READ  {', '.join(files[:3])}"
                     + (f" (+{len(files)-3} more)" if len(files) > 3 else ""))
            show_details(label, result.get("output", ""), open=False)

        elif result["type"] in ("glob", "grep"):
            items = result.get("files") or result.get("matches") or []
            show_details(
                f"🔍  {result['type'].upper()}  {len(items)} result(s)",
                "\n".join(items),
                open=False,
            )

        else:  # message / other
            show_details(f"⚙️  {result['type'].upper()}", result.get("output", ""), open=False)

        log.append({**step, "result": result, "accepted": accepted})

    # ── 3. Verify ─────────────────────────────────────────────────────────────
    accepted_writes = [s for s in log if s["result"]["type"] == "write" and s["accepted"]]
    if accepted_writes:
        show_rule("Verification")
        all_ok = True
        for s in accepted_writes:
            path = Path(repo_root) / s["result"]["file_path"]
            icon = "✅" if path.exists() else "❌"
            display(HTML(f'<span style="font-size:0.82rem">{icon} {s["result"]["file_path"]}</span>'))
            if not path.exists():
                all_ok = False
        msg = "All files confirmed on disk." if all_ok else "Some files are missing."
        display(HTML(f'<span style="font-size:0.82rem;color:#57606a">{msg}</span>'))

    show_rule("Loop complete")
    return log

### 14.3 — What didn't change

`plan_task()` and `execute_step()` are untouched.

`plan_task()` picked up the `URL:` field automatically because `register_action()` added `"URL"` to the registry's field list, which `plan_task()` reads at parse time.

`execute_step()` dispatched to `_handle_fetch` automatically because the handler was stored in `ACTION_REGISTRY["fetch"]`.

This is the registry's payoff: a new capability required **one** new function and **one** registration call, not edits to three separate functions.


### 14.4 Try it — run the agent with a URL task

The cell below asks the agent to fetch the Python `pathlib` docs and write a one-page cheat sheet based on them. The plan should come back with a `fetch` step followed by a `write` step — no reads needed.


In [ ]:
# ask for a new sheet
log = agent_loop(
    task      = ("Fetch https://docs.python.org/3/library/pathlib.html "
                 "and display a concise cheat sheet to docs/pathlib_cheatsheet.md"),
    repo_root = REPO_ROOT,
)

In [ ]:
log = agent_loop(
    task      = "Use readxml to fetch https://rss.slashdot.org/Slashdot/slashdotMain "
                "and write the top 5 headlines to docs/slashdot_headlines.md with a one line summary each",
    repo_root = "sample_project"
)

**Exercise B — add a new retrieval strategy: recency**

Some tasks are best answered by the files changed most recently — e.g. "what did I just break?". Add a `recent` strategy that ranks files by modification time instead of relevance score.


In [ ]:
# Exercise B — recency retrieval strategy
#
# Step 1: write the function
def recent_retrieve(repo_root: str = REPO_ROOT, top_k: int = 8) -> list[dict]:
    """Return the top-k most recently modified source files."""
    files = glob_files("*", repo_root=repo_root)
    files = [f for f in files if Path(f["path"]).suffix.lower() in CODE_EXTENSIONS]
    # TODO: sort by modification time and return top_k
    # Hint: (Path(repo_root) / f["path"]).stat().st_mtime
    raise NotImplementedError("complete this function")


# Step 2: wire into pick_strategy()
# Add a detection rule — if the query mentions words like "recent", "last",
# "changed", "broke", "latest", route to "recent".
#
# In pick_strategy(), add at the TOP (before the grep/semantic checks):
#   if any(w in q for w in ("recent", "last", "changed", "broke", "latest")):
#       return "recent"
#
# Keep the existing default: return "fuzzy"
#
# Then in run(), add a branch before the else clause:
#   elif strat == "recent":
#       candidates = recent_retrieve(repo_root=repo_root, top_k=top_k)


# Step 3: test it
# Uncomment once you've implemented the function and wired it in:
# result = run("what files changed most recently?", repo_root=REPO_ROOT)
# show_reply(result.reply)
# print("Strategy:", result.strategy)


In [ ]:
# Exercise B — Solution

# ── Step 1: complete recent_retrieve() ───────────────────────────────────────
def recent_retrieve(repo_root: str = REPO_ROOT, top_k: int = 8) -> list[dict]:
    """Return the top-k most recently modified source files."""
    files = glob_files("*", repo_root=repo_root)
    files = [f for f in files if Path(f["path"]).suffix.lower() in CODE_EXTENSIONS]
    files.sort(
        key=lambda f: (Path(repo_root) / f["path"]).stat().st_mtime,
        reverse=True,
    )
    return files[:top_k]


# ── Step 2: redefine pick_strategy() with recency detection ──────────────────
def pick_strategy(query: str) -> str:
    """
    Classify *query* into one of four retrieval strategies:
      "recent"   — query is about recently changed files
      "grep"     — looking for a specific symbol or string in code
      "semantic" — conceptual / meaning-based question
      "fuzzy"    — everything else (filename-level search, default)
    """
    q = query.lower()
    # Recency check first — most specific signal
    if any(w in q for w in ("recent", "last", "changed", "broke", "latest")):
        return "recent"
    # Delegate the rest to the original grep/semantic/fuzzy logic
    if _GREP_SIGNALS.search(query):
        return "grep"
    words = set(q.split())
    if words & _CONCEPTUAL_WORDS:
        return "semantic"
    return "fuzzy"   # default — fast, good enough for named-file queries


# ── Step 3: redefine run() with recency branch ───────────────────────────────
# We extend the existing run() rather than replacing it, so all other strategies
# continue to go through the full pipeline (budget_load, compact, show_panel).
def run(
    query:     str,
    repo_root: str = REPO_ROOT,
    strategy:  str = "auto",
    top_k:     int = 8,
) -> RunResult:
    manifest     = load_manifest(repo_root)
    manifest_tok = manifest["tokens"]
    strat        = pick_strategy(query) if strategy == "auto" else strategy

    if strat == "recent":
        candidates = recent_retrieve(repo_root=repo_root, top_k=top_k)
    elif strat == "grep":
        candidates = grep_rank(query, repo_root=repo_root)
        found = {f["path"] for f in candidates}
        extras = [f for f in rank_files(glob_files("*.py", repo_root=repo_root), query)
                  if f["path"] not in found]
        candidates = (candidates + extras)[:top_k]
    elif strat == "semantic":
        candidates = semantic_retrieve(query, repo_root=repo_root, top_k=top_k)
    else:   # fuzzy (and any unrecognised strategy)
        candidates = rank_files(glob_files("*.py", repo_root=repo_root), query)[:top_k]

    loaded = budget_load(candidates, already_used=manifest_tok, repo_root=repo_root)
    hot_tok = sum(f.get("tokens", 0) for f in loaded if f["hot"])
    total   = manifest_tok + hot_tok + count_tokens(query)
    loaded, total, compact_log = compact(loaded, query, total)

    hot_files = [f for f in loaded if f["hot"]]
    reply, tokens_used = ask_with_manifest(query, repo_root=repo_root, files=hot_files)

    return RunResult(
        reply=reply, strategy=strat, files=loaded,
        tokens_used=tokens_used, compact_log=compact_log,
    )


In [ ]:
# ── Step 4: test it ───────────────────────────────────────────────────────────
result = run("what files changed most recently?", repo_root="sample_project")
show_reply(result.reply)
print("Strategy used:", result.strategy)

**Exercise C — add a new file type: `.sql`**

The agent currently ignores `.sql` files. One line change to `CODE_EXTENSIONS` fixes that — but you also need to clear the embed cache so the index is rebuilt with the new files included.


In [ ]:
# Exercise C — index .sql files

# Step 1: add the extension
CODE_EXTENSIONS.add(".sql")
print("Extensions now include:", sorted(CODE_EXTENSIONS))

# Step 2: the embed index is cached per repo_root — clear it so the
# next semantic_retrieve() call picks up any .sql files it finds.
_EMBED_INDEX.clear()
print("Embed index cleared — will rebuild on next semantic query.")

# Step 3: create a tiny SQL file and verify it gets indexed
import os
os.makedirs("sample_project", exist_ok=True)
Path("sample_project/schema.sql").write_text(
    "CREATE TABLE users (id INTEGER PRIMARY KEY, name TEXT, email TEXT);\n"
    "CREATE TABLE orders (id INTEGER PRIMARY KEY, user_id INTEGER, total REAL);\n"
)

idx = build_index(repo_root="sample_project")
sql_entries = [e for e in idx if e["path"].endswith(".sql")]
print(f"\nSQL files in index: {[e['path'] for e in sql_entries]}")


In [ ]:
# Exercise C — Solution: verify .sql files are indexed and retrieved semantically

idx     = build_index(repo_root="sample_project")
results = semantic_retrieve(
    "what tables are defined in the database schema?",
    repo_root="sample_project",
)

print("SQL files in index:", [e["path"] for e in idx if e["path"].endswith(".sql")])
print("\nSemantic retrieve results:")
for r in results:
    print(f"  {r['path']}")


---

## Chapter 15 — Reflection

**Goal:** teach the agent to observe each step's result before deciding what to do next, rather than blindly executing a fixed plan.

Up to now `agent_loop` plans everything upfront and executes each step without looking at the output. That works well for predictable tasks ("add type annotations to these three files") but breaks down when:

- A fetch step returns an error and subsequent write steps are now pointless.
- A bash step reveals the file doesn't exist and the plan needs to change.
- A read step returns "I don't know" and more context is needed before writing.

A *reflecting* agent pauses after each step, reads the result, and asks: *"Given what just happened, does the remaining plan still make sense?"* This is the **ReAct pattern** (Reason + Act) — the model reasons about observations before taking the next action.

We saw a hand-written version of this in `test_loop` (Chapter 13): after each pytest run, the fix instruction was built from the actual failure output. Chapter 15 generalises that pattern into `agent_loop` itself.


### 15.1 `reflect()` — should we continue?

After each step, we call `reflect()` with the task, the step that just ran, its result, and the remaining plan. The LLM returns one of three decisions:

- **continue** — result looks good, proceed with the next step as planned
- **revise** — result changes things; here is a revised remaining plan
- **abort** — something went wrong; stop and report

The function returns a dict so the caller can branch on `decision` and optionally get a `revised_steps` list and a human-readable `reason`.


In [ ]:
_REFLECT_SYSTEM = """\
You are reviewing the result of one step in an agent's plan.

Decide whether the agent should:
  CONTINUE  — the result looks good; proceed with the remaining steps as planned.
  REVISE    — the result changes things; output a revised plan for the remaining steps.
  ABORT     — something went wrong and the task cannot proceed; explain why.

Reply in exactly this format:

DECISION: <continue|revise|abort>
REASON: <one sentence>
REVISED_STEPS:
<only present when DECISION is revise — list revised remaining steps in the same
STEP/ACTION/TARGET/INSTRUCTION/CMD/URL/MESSAGE format as the original plan>
"""


# ── Shared parser ─────────────────────────────────────────────────────────────

def _parse_reflect_raw(raw: str) -> dict:
    """
    Parse the LLM's DECISION/REASON/REVISED_STEPS response into a structured dict.

    This helper is called by both reflect() (standalone) and the in-thread
    reflection inside agent_loop_reflect — keeping the parsing logic in one place.

    Returns:
        {"decision": "continue"|"revise"|"abort",
         "reason":   str,
         "revised_steps": list[dict] | None}   # None unless decision == "revise"
    """
    decision = "continue"
    reason   = ""
    revised  = None

    # Pull out DECISION and REASON from the first two tagged lines.
    for line in raw.splitlines():
        if line.startswith("DECISION:"):
            decision = line.split(":", 1)[1].strip().lower()
        elif line.startswith("REASON:"):
            reason = line.split(":", 1)[1].strip()

    if decision == "revise":
        # Everything after "REVISED_STEPS:" is a new plan in the same
        # STEP/ACTION/... block format used by plan_task() — reuse the splitter.
        after   = raw.split("REVISED_STEPS:", 1)[-1].strip()
        revised = []
        for block in re.split(r"(?m)(?=^STEP:)", after):
            if not block.strip():
                continue
            # The walrus operator  (m := ...)  assigns the regex match to m
            # inside the comprehension condition, so we can use m.group() in
            # the key/value part of the same expression without a second match call.
            fields = {
                m.group(1).lower(): m.group(2).strip()
                for line in block.splitlines()
                if (m := re.match(
                    # Build the field pattern from the registry so it stays
                    # in sync when new actions are added.
                    rf"^({'|'.join(sorted({'STEP','ACTION'} | {f for a in ACTION_REGISTRY.values() for f in a['fields']}))}):\s*(.+)$",
                    line, re.IGNORECASE
                ))
            }
            if "action" in fields:
                # Derive the field set from the registry so new actions
                # are included automatically.
                all_rev_fields = {"step", "action"} | {
                    f.lower() for a in ACTION_REGISTRY.values() for f in a["fields"]
                }
                revised.append({k: fields.get(k, "") for k in all_rev_fields})

    return {"decision": decision, "reason": reason, "revised_steps": revised}


# ── Standalone reflect() ──────────────────────────────────────────────────────

def reflect(
    task:           str,
    step:           dict,
    result:         dict,
    remaining_plan: list[dict],
) -> dict:
    """
    Ask the LLM whether to continue, revise, or abort after a step.

    This is the *standalone* version: it opens a fresh two-message conversation
    (system + user) and has no memory of previous steps.  It is useful for
    testing reflection logic in isolation.

    agent_loop_reflect uses an *in-thread* variant instead: the reflection
    prompt is appended to the shared conversation thread so the model can see
    the full execution history when it reasons.  Both variants call the same
    _parse_reflect_raw() to interpret the response.

    Returns:
        {"decision": "continue"|"revise"|"abort",
         "reason":   str,
         "revised_steps": list[dict] | None}
    """
    # ── Build a plain-text summary of the result ──────────────────────────────
    result_text = (
        f"type={result.get('type')}  "
        f"ok={result.get('ok', '?')}  "
        f"output={str(result.get('output', ''))}"
    )

    # ── Format the remaining steps the LLM will potentially revise ───────────
    remaining_text = "\n\n".join(
        f"STEP: {s['step']}\nACTION: {s['action']}\n" +
        (f"TARGET: {s['target']}"   if s.get('target')  else "") +
        (f"CMD: {s['cmd']}"         if s.get('cmd')     else "") +
        (f"URL: {s['url']}"         if s.get('url')     else "") +
        (f"MESSAGE: {s['message']}" if s.get('message') else "")
        for s in remaining_plan
    ) or "(none — this was the last step)"

    # ── Call the LLM ──────────────────────────────────────────────────────────
    raw, _ = chat([
        {"role": "system", "content": _REFLECT_SYSTEM},
        {"role": "user",   "content": (
            f"TASK: {task}\n\n"
            f"STEP JUST EXECUTED:\n"
            f"  action: {step['action']}\n"
            f"  description: {step['step']}\n\n"
            f"RESULT:\n{result_text}\n\n"
            f"REMAINING PLAN:\n{remaining_text}"
        )},
    ])

    # ── Parse and return ──────────────────────────────────────────────────────
    return _parse_reflect_raw(raw)


### 15.2 `agent_loop` with reflection

The loop is identical to Chapter 14 except for one thing: after each step, we call `reflect()` and branch on the decision. If the LLM says `revise`, the remaining steps are replaced with the revised plan before continuing. If it says `abort`, we stop immediately.

Note that reflection adds one LLM call per step. For a 4-step plan that's 4 extra calls. This is the cost of adaptability — worth it for tasks where step results are unpredictable (fetching URLs, running tests, calling external APIs), less necessary for mechanical tasks like adding type annotations to known files.


In [ ]:
# ── Thread compaction ─────────────────────────────────────────────────────────

def _compact_step_turns(step_turns: list[dict]) -> dict:
    """
    Summarise one completed step's turn-group into a single assistant message.

    step_turns is the slice of messages that belong to one step:
      [user "STEP N: …", assistant result, user reflection prompt, assistant decision]

    The summary is prefixed with [COMPACTED] so compact_thread() can skip
    already-compacted entries on subsequent passes.
    """
    content  = "\n\n".join(t["content"] for t in step_turns)
    summary, _ = chat([
        {"role": "system", "content": (
            "Summarise this completed agent step in 1-2 sentences: "
            "what action was taken, what was the key result, and what was decided "
            "(continue / revise / abort). Be concise."
        )},
        {"role": "user", "content": content},
    ])
    return {"role": "assistant", "content": f"[COMPACTED] {summary}"}


def compact_thread(messages: list[dict], remaining: list[dict]) -> list[dict]:
    """
    Collapse completed step turns in the shared thread to save token budget.

    Conservative rule: as long as any write or bash steps remain in the plan,
    keep all content live — those steps may need prior fetch/bash/read outputs
    as source material.  Once no write/bash steps remain, each completed step's
    turn-group is replaced with a one-line summary from _compact_step_turns().

    The system prompt (messages[0]) is never compacted.
    """
    # Keep everything live while destructive steps remain in the plan.
    if any(s["action"] in ("write", "bash") for s in remaining):
        return messages

    compacted = [messages[0]]   # system prompt always survives intact
    i = 1
    while i < len(messages):
        msg = messages[i]
        # Step turn-groups always start with a "STEP N:" user message
        # (injected by the loop before each execute_step call).
        if msg["role"] == "user" and msg["content"].startswith("STEP "):
            # Gather every message until the next "STEP N:" boundary.
            step_turns = [msg]
            i += 1
            while i < len(messages) and not (
                messages[i]["role"] == "user"
                and messages[i]["content"].startswith("STEP ")
            ):
                step_turns.append(messages[i])
                i += 1
            compacted.append(_compact_step_turns(step_turns))
        else:
            compacted.append(msg)
            i += 1
    return compacted


# ── Loop helpers ──────────────────────────────────────────────────────────────

def _write_context_from_thread(thread: list[dict]) -> str:
    """
    Scan the shared thread for prior fetch / bash / read outputs and return
    them as a single block of text.

    This is injected into write-step instructions so the LLM has real content
    to summarise or transform — without it the model has to invent the content.

    We only look at non-compacted assistant turns because compacted turns have
    already lost the detailed output (replaced with a 1-2 sentence summary).
    The keywords FETCHED / RAN / READ match the prefixes that _display_result()
    writes into result_text, so the two functions are deliberately coupled.
    """
    chunks = [
        t["content"] for t in thread
        if t["role"] == "assistant"
        and not t["content"].startswith("[COMPACTED]")
        and any(kw in t["content"] for kw in ("FETCHED", "RAN", "READ"))
    ]
    return "\n\n".join(chunks)


def _display_result(result: dict, repo_root: str, dry_run: bool = False) -> tuple[str, bool]:
    """
    Display the result of execute_step() as a collapsible <details> block.

    Every intermediate output is shown collapsed by default so the notebook
    stays compact.  The caller (agent_loop_reflect) is responsible for
    rendering the final verdict at normal size via show_reply().

    Write steps are handled automatically — no user prompt:
      - dry_run=True  : show diff collapsed, do not write.
      - dry_run=False : write the file, show diff collapsed.

    BASH failures and live WRITE diffs start expanded so errors / changes
    are immediately visible without a click.

    Returns:
        result_text — plain-text summary appended to the shared thread.
                      Fetch / bash / read results are prefixed with FETCHED /
                      RAN / READ so _write_context_from_thread() can find them.
        accepted    — True only when a write step was confirmed and written.
    """
    accepted = False
    rtype    = result["type"]

    if rtype == "write":
        diff      = result.get("diff", "")
        file_path = result["file_path"]
        if not diff:
            # apply_patch returned no changes — nothing to write.
            show_details(
                f"⚠️  WRITE  {file_path}  — no changes proposed",
                "The model returned the same content as the original file.\n"
                "The instruction or context may need to be more specific.",
                open=True,
            )
            result_text = f"WRITE {file_path} — no changes proposed"
        elif dry_run:
            show_details(f"✏️  WRITE  {file_path}  (dry run — not written)", diff, open=False)
            result_text = f"(dry run) Would write {file_path}"
        else:
            # Autonomous mode: write immediately, show the diff collapsed.
            path = Path(repo_root) / file_path
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_text(result["new_content"])
            accepted = True
            show_details(f"✏️  WRITE  {file_path}  ✅ written", diff, open=False)
            result_text = f"Wrote {file_path}"

    elif rtype == "fetch":
        ok      = result.get("ok", True)
        icon    = "✅" if ok else "❌"
        preview = result["output"][:FETCH_DISPLAY_CHARS]
        if len(result["output"]) > FETCH_DISPLAY_CHARS:
            preview += f"\n\n[… {len(result['output']) - FETCH_DISPLAY_CHARS} more chars not shown]"
        show_details(f"{icon}  FETCH  {result['url']}", preview, open=not ok)
        # FETCHED prefix lets _write_context_from_thread() recognise this later.
        # Full output (not preview) is kept so write steps have the real content.
        result_text = f"FETCHED {result['url']}:\n{result['output']}"

    elif rtype == "bash":
        ok   = result.get("ok", True)
        icon = "✅" if ok else "❌"
        out  = result.get("output", "")
        # Expand on failure so errors are immediately visible without a click.
        show_details(f"{icon}  BASH  `{result['cmd']}`", out[:2000], open=not ok)
        result_text = f"RAN `{result['cmd']}` :\n{out}"

    elif rtype == "read":
        files = result.get("files", [])
        label = (f"📖  READ  {', '.join(files[:3])}"
                 + (f" (+{len(files)-3} more)" if len(files) > 3 else ""))
        show_details(label, result.get("output", ""), open=False)
        result_text = f"READ:\n{result.get('output', '')}"

    elif rtype in ("glob", "grep"):
        items = result.get("files") or result.get("matches") or []
        label = f"🔍  {rtype.upper()}  {len(items)} result(s)"
        show_details(label, "\n".join(items), open=False)
        result_text = f"{rtype.upper()}: {', '.join(items)}"

    else:   # any other action types
        out = result.get("output", "")
        show_details(f"⚙️  {rtype.upper()}", str(out), open=False)
        result_text = str(out)

    return result_text, accepted


# ── Shared-thread agent loop ──────────────────────────────────────────────────

def agent_loop_reflect(
    task:       str,
    repo_root:  str  = REPO_ROOT,
    reflect_on: set  = {"fetch", "bash"},
    max_steps:  int  = 20,
    dry_run:    bool = False,
) -> list[dict]:
    """
    ReAct-style agent loop: plan → (execute → display → reflect → compact) → verify.

    Five differences from the Chapter 14 agent_loop
    ────────────────────────────────────────────────
    1. Shared thread — every LLM call appends to the same `thread` list, so each
       call sees the full history: planning, execution, reflection, decisions.

    2. Per-step reflection — after each step whose action is in `reflect_on`,
       the model reads the result and returns CONTINUE / REVISE / ABORT.
       On REVISE, the remaining plan is swapped out on the spot.

    3. Write-step context — before generating file content, the loop scans the
       thread for prior FETCHED / RAN / READ outputs and injects them as source
       material.  This prevents the model from inventing content it never saw.

    4. Thread compaction — once no write/bash steps remain, completed step
       turn-groups are collapsed to 1-2 sentence summaries to keep the thread
       from growing without bound on long tasks.

    5. Final verification — a closing LLM call reads the whole thread and gives
       a holistic verdict on whether the original task was accomplished.

    Parameters
    ----------
    dry_run : if True, write steps show their diff but never prompt or write to disk.
              Useful for previewing what the agent would do without side effects.
    """
    show_rule(f"Agent Loop (reflect)  ·  {task[:55]}")
    print(f"repo: {repo_root}  dry_run={dry_run}\n")

    # ── 1. Seed the shared thread ─────────────────────────────────────────────
    # The system prompt bundles three things: the task, the project manifest
    # (so the model knows what files exist), and _PLAN_SYSTEM (so that when it
    # writes REVISED_STEPS it uses the correct ACTION vocabulary for this chapter).
    manifest = load_manifest(repo_root)
    thread   = [{
        "role": "system",
        "content": (
            f"You are a coding agent executing this task:\n{task}\n\n"
            f"Project context:\n{manifest['text']}\n\n"
            f"Available actions (use these formats when revising the plan):\n"
            f"{_PLAN_SYSTEM}"
        )
    }]

    # ── 2. Plan ───────────────────────────────────────────────────────────────
    plan = plan_task(task, repo_root=repo_root)
    if not plan:
        print("Planning failed — no steps parsed.")
        return []

    plan_text = "\n".join(
        f"  {i+1}. {s['action'].upper()} — {s['step']}"
        for i, s in enumerate(plan)
    )
    # Record the agreed plan in the thread so later reflection calls can
    # reference it when explaining why they want to revise.
    thread.append({"role": "user",      "content": f"Plan:\n{plan_text}"})
    thread.append({"role": "assistant", "content": "Understood. Executing."})
    print(f"Plan: {len(plan)} step(s)\n")

    log       = []
    remaining = list(plan)

    # ── 3. Execute steps ──────────────────────────────────────────────────────
    while remaining:
        step     = remaining.pop(0)
        step_num = len(log) + 1
        show_rule(f"Step {step_num}  {step['action'].upper()}  {step['step']}")

        # Open each step with a "STEP N:" user turn.  compact_thread() uses
        # this marker to identify turn-group boundaries when compacting.
        thread.append({"role": "user", "content": (
            f"STEP {step_num}: {step['action'].upper()} — {step['step']}"
            + (f"\ntarget: {step['target']}"           if step.get('target')      else "")
            + (f"\ninstruction: {step['instruction']}" if step.get('instruction') else "")
            + (f"\ncmd: {step['cmd']}"                 if step.get('cmd')         else "")
            + (f"\nurl: {step['url']}"                 if step.get('url')         else "")
            + (f"\nmessage: {step['message']}"         if step.get('message')     else "")
        )})

        # 3a. Context injection for write steps ───────────────────────────────
        # Pull any fetch / bash / read outputs from the thread and append them
        # to the instruction.  Without this, the write LLM call has no access
        # to the content it is supposed to be summarising or transforming.
        if step["action"] == "write":
            context = _write_context_from_thread(thread)
            if context:
                step = {**step, "instruction": (
                    step["instruction"] +
                    "\n\nContext from previous steps — use as source material:\n\n" +
                    context
                )}

        # 3b. Execute and display ─────────────────────────────────────────────
        result                = execute_step(step, repo_root=repo_root)
        result_text, accepted = _display_result(result, repo_root, dry_run=dry_run)
        # Record the result in the thread so reflection (and future steps) see it.
        thread.append({"role": "assistant", "content": result_text})

        # 3c. Reflect ─────────────────────────────────────────────────────────
        # Reflect on steps with unpredictable outcomes (fetch/bash by default).
        # Always reflect — even on the last step — so the model can add more
        # fix iterations if tests are still failing.
        reflection = None
        if step["action"] in reflect_on and len(log) < max_steps:
            show_rule("Reflecting…")

            remaining_txt = (
                "; ".join(f"{s['action'].upper()} {s['step']}" for s in remaining)
                or "none — but you may add more steps if needed"
            )

            # Build context-aware guidance.  When a bash step fails (non-zero
            # exit or test failures), spell out that CONTINUE is wrong here.
            bash_failed = step["action"] == "bash" and not result.get("ok", True)
            failure_hint = (
                "\n\nIMPORTANT: The bash step returned errors or test failures. "
                "CONTINUE is only correct when everything succeeded. "
                "If any test is still failing you MUST choose REVISE and add "
                "WRITE + BASH steps to fix the issue."
            ) if bash_failed else ""

            # Append the reflection request to the *shared thread* so the model
            # reasons with full history, not just the isolated step result.
            thread.append({"role": "user", "content": (
                f"Reflect on this result. Remaining steps: {remaining_txt}"
                f"{failure_hint}\n\n"
                "Reply with:\n"
                "DECISION: <continue|revise|abort>\n"
                "REASON: <one sentence>\n"
                "REVISED_STEPS:\n"
                "<if revising or adding steps — use STEP/ACTION/... format>"
            )})
            raw, _ = chat(thread)
            thread.append({"role": "assistant", "content": raw})

            # Reuse the same parser as the standalone reflect() above.
            reflection = _parse_reflect_raw(raw)
            icon = {"continue": "✅", "revise": "🔄", "abort": "🛑"}.get(
                reflection["decision"], "❓"
            )
            print(f"{icon} {reflection['decision'].upper()}  —  {reflection['reason']}")

            if reflection["decision"] == "abort":
                log.append({**step, "result": result, "accepted": accepted,
                             "reflection": reflection})
                show_rule("Aborted")
                break
            elif reflection["decision"] == "revise" and reflection["revised_steps"]:
                remaining = reflection["revised_steps"]
                print(f"\nRevised plan: {len(remaining)} remaining step(s)")

        log.append({**step, "result": result, "accepted": accepted,
                    "reflection": reflection})

        # 3d. Compact ─────────────────────────────────────────────────────────
        # Once no write/bash steps remain, prior step turns are safe to collapse.
        # This keeps the thread from growing without bound on long tasks.
        thread = compact_thread(thread, remaining)

    # ── 4. Final verification ─────────────────────────────────────────────────
    # A closing LLM call reads the whole thread and judges whether the original
    # task was actually accomplished — not just whether the steps ran without error.
    show_rule("Final Verification")
    thread.append({"role": "user", "content": (
        f"The loop is complete. Was the original task fully accomplished?\n\n"
        f"Task: {task}\n\n"
        f"Review everything above and give a concise verdict: "
        f"what was done correctly, what (if anything) is missing or wrong."
    )})
    verdict, _ = chat(thread)
    thread.append({"role": "assistant", "content": verdict})
    show_reply(verdict)

    show_rule("Done")
    return log

### 15.3 Try it

The task below fetches a URL and then writes a summary to a file. Because `fetch` is in `reflect_on`, the agent will pause after downloading the page and decide whether the write step still makes sense based on what it got back.

Try replacing the URL with a broken one (`https://this-does-not-exist.xyz`) and watch the agent abort rather than writing an empty or error-filled file.

> **Model note:** reflection requires the LLM to read a result and make a reasoned decision.
> `gemini-2.5-flash-lite` handles this inconsistently — it occasionally says CONTINUE when it
> should REVISE, or produces a write step with no actual changes. If that happens, re-run the cell
> or switch to a stronger model. The logic is correct; the model is just underpowered for the task.

In [ ]:
# run loop
log = agent_loop_reflect(
    task = (
        "Use readxml to fetch https://rss.slashdot.org/Slashdot/slashdotMain "
        "and write a one-paragraph summary of today's top stories to docs/slashdot_summary.md."
    ),
    repo_root = REPO_ROOT,
)

### 15.4 Your turn — exercises

**Exercise A — reflect on all steps**

Change `reflect_on` to `{"fetch", "bash", "read", "write"}` and run the same task. Notice how many extra LLM calls are made and whether the decisions add value for the read and write steps.

**Exercise B — collapse `test_loop` into `agent_loop_reflect`**

`test_loop` (Chapter 13) is a hand-written feedback loop: generate tests → run → fix → retry. Rewrite it as a call to `agent_loop_reflect`:

1. The initial plan is two steps: `write` (generate tests) + `bash` (run pytest).
2. After the bash step, reflection checks the pytest output and either says `continue` (all passed) or `revise` (add a new write step to fix the failures).
3. The loop naturally retries until tests pass or the LLM gives up.

This is the generalisation we promised in Chapter 13.

> **⚠️ This exercise is genuinely hard for weaker models.**
> It asks the LLM to write correct pytest code, run it, read the failure output, understand what
> went wrong, and write a fix — all in a loop. `gemini-2.5-flash-lite` will often get partway
> there and then stall: producing tests that don't quite pass, or fixes that don't change anything.
>
> **This is not a bug.** The reflect/revise logic is working correctly — it's the model that's
> hitting its limits. For this exercise, `qwen3-coder-next` (via Ollama, free cloud tier) or any other coder models, they will usually give you a much cleaner experience.
>
> The solution cell below shows what a successful run looks like so you know what to aim for.

**Exercise C — add a `confidence` field to reflect()**

Extend the `_REFLECT_SYSTEM` prompt and the `reflect()` parser to return a `confidence` score (0.0–1.0) alongside the decision. Use it to skip reflection when confidence is high (e.g. `> 0.9`) to save LLM calls on obvious steps.

In [ ]:
# Exercise B — Solution
#
# The hand-written test_loop (Chapter 13) did three things explicitly:
#   1. Write a test file with generate_tests()
#   2. Run pytest and capture the output
#   3. If failures: build a fix instruction from the live output, rewrite, retry
#
# With agent_loop_reflect, all three are expressed as a task description.
# The agent plans: write → bash.  After each bash step, reflect() reads the
# pytest output and decides:
#   - CONTINUE  (all tests passed — nothing left to do)
#   - REVISE    (tests failed — add another write + bash pair to fix them)
#   - ABORT     (failures it can't fix — give up and report)
#
# The retry loop is no longer explicit code.  It emerges from the reflect/revise
# cycle, and the LLM decides when to stop — exactly what test_loop did manually.
#
# ── Model note ────────────────────────────────────────────────────────────────
# This task requires multi-step reasoning: write tests → read failure output →
# understand the error → write a correct fix.  gemini-2.5-flash-lite will
# sometimes succeed and sometimes stall (writing fixes that change nothing, or
# tests that never quite pass).  If you hit that wall, switch to:
#   - Ollama: qwen3-coder-next:cloud  (free, no GPU needed)
#   - Colab:  gemini-2.5-flash        (uncomment in the setup cell)
# ─────────────────────────────────────────────────────────────────────────────

log = agent_loop_reflect(
    task = (
        "Generate a complete pytest class test file with a test class for `utils/formatter.py` "
        "and save it as `tests/test_formatter_reflect.py`. "
        "Then run it with: python3 -m pytest tests/test_formatter_reflect.py -v\n"
        "If any tests fail, fix the test file (do NOT modify the source module) "
        "and re-run pytest. Repeat until all tests pass or you are unable to fix them. "
    ),
    repo_root  = "sample_project",
    reflect_on = {"bash"},   # reflect after every pytest run, not after writes
)

In [ ]:
show_log(log)

---

## Architecture Map

The agent's core loop in one diagram:

```
query / task
    │
    ▼
plan_task()                     ← LLM + ACTION_REGISTRY system prompt
    │  returns list[dict] steps
    │
    ▼  for each step:
execute_step()
    │
    ├─ ACTION: read    → jit_read()       → file content
    ├─ ACTION: glob    → glob_files()     → file list
    ├─ ACTION: grep    → grep_repo()      → matching lines
    ├─ ACTION: fetch   → fetch_url()      → stripped HTML
    ├─ ACTION: readxml → _xml_to_text()   → clean plain text
    ├─ ACTION: bash    → tool_bash()      → stdout / stderr
    └─ ACTION: write   → apply_patch()    → diff + new content
    │
    ▼
_display_result()               ← collapsible panel per step
    │
    ▼  (Chapter 15 only)
reflect()                       ← CONTINUE / REVISE / ABORT
    │
    ├─ CONTINUE → next step
    ├─ REVISE   → re-plan from current state
    └─ ABORT    → stop with reason
    │
    ▼
final verify step  (bash)
    │
    ▼
show_log(log)                   ← full run summary
```

**Context management runs in parallel with every step:**
- `count_tokens()` tracks usage after each action
- `compact()` evicts cold files when the prompt exceeds 80 % of `TOKEN_BUDGET`
- `_compact_step_thread()` (Chapter 15) summarises the running step thread itself


---

## Epilogue — From Scratch to LlamaIndex

You've now built by hand what [LlamaIndex](https://docs.llamaindex.ai) packages into a library.

Here's the same retrieval pipeline — Chapters 5 through 9 — in ten lines. Every comment maps back to a function you wrote.

| Capability | Your code | LlamaIndex |
|---|:---:|:---:|
| Chat with a local LLM | ✅ | ✅ |
| File loading and indexing | ✅ | ✅ |
| Semantic (embedding-based) retrieval | ✅ | ✅ |
| Agent loop with tool execution | ✅ | ✅ |
| Token budget awareness, HOT/COLD splitting | ✅ | ❌ |
| Fuzzy filename ranking + grep retrieval | ✅ | ❌ |
| Microcompaction for long sessions | ✅ | ❌ |
| Structured text plans (`STEP:/ACTION:/TARGET:`) | ✅ | ❌ |
| Reflect / revise loop with thread compaction | ✅ | ❌ |
| Scales to thousands of files | ⚠️ | ✅ |
| Persistent vector store (survives restarts) | ⚠️ | ✅ |
| 150+ data connectors (PDF, Notion, S3…) | ❌ | ✅ |

⚠️ = works for small-to-medium repos, not designed for production scale

LlamaIndex's `ReActAgent` and yours solve the same problem at different levels of abstraction. The gap between *using* LlamaIndex and *understanding* it is exactly what these chapters covered.

In [ ]:
import subprocess, sys

# ── Install LlamaIndex + the right backend packages ───────────────────────────
# llama-index-readers-file is needed for SimpleDirectoryReader to handle
# .py / .md / .txt files without printing warnings about missing readers.

_LLAMA_DEPS = ["llama-index", "llama-index-readers-file"]

if EMBED_BACKEND == "sentence-transformers":
    _LLAMA_DEPS.append("llama-index-embeddings-huggingface")
else:
    _LLAMA_DEPS += ["llama-index-llms-ollama", "llama-index-embeddings-ollama"]

for _pkg in _LLAMA_DEPS:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg],
                          stdout=subprocess.DEVNULL)
print("LlamaIndex ready.")

In [ ]:
# Five chapters of retrieval, ten lines of LlamaIndex.
# Every comment maps back to the function you wrote that does the same thing.

import logging, warnings

# Suppress noisy startup logs from LlamaIndex, HuggingFace, and transformers.
# None of these indicate actual errors — they're informational noise.
logging.getLogger("llama_index").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings
from llama_index.core.node_parser import SentenceSplitter

# ── Chapter 2: which model / embed to use ────────────────────────────────────
if BACKEND == "ollama":
    from llama_index.llms.ollama import Ollama
    from llama_index.embeddings.ollama import OllamaEmbedding
    Settings.llm         = Ollama(model=_ollama.model, request_timeout=120)
    Settings.embed_model = OllamaEmbedding(model_name=_ollama.embed)
else:
    # Colab: HuggingFace embeddings — same model loaded in Chapter 9.
    # LlamaIndex has no direct Colab AI integration, so we do retrieval only
    # (no LLM generation step) and show the top-ranked chunks instead.
    from llama_index.embeddings.huggingface import HuggingFaceEmbedding
    Settings.embed_model = HuggingFaceEmbedding(model_name="all-MiniLM-L6-v2")
    Settings.llm         = None   # retrieval only — suppresses MockLLM warning below
    import llama_index.core
    llama_index.core.Settings._llm = None   # prevent MockLLM fallback message

# ── Chapter 5: walk the repo and load files (our glob_files() + jit_read()) ──
documents = SimpleDirectoryReader(
    "sample_project",
    recursive     = True,
    required_exts = [".py", ".md", ".txt", ".sql"],
).load_data()

# ── Chapter 9: chunk, embed, and index (our build_index()) ───────────────────
index = VectorStoreIndex.from_documents(
    documents,
    transformations=[SentenceSplitter(chunk_size=512, chunk_overlap=64)],
)

QUERY = "Where is the nesting logic handled?"

# ── Chapters 4 + 9: retrieve and answer ──────────────────────────────────────
if BACKEND == "ollama":
    # Full RAG: retrieve relevant chunks, then generate an answer with the LLM.
    engine   = index.as_query_engine(similarity_top_k=4)
    response = engine.query(QUERY)
    print(f"Query: {QUERY}\n")
    print(response)
else:
    # Colab (retrieval only): return the top-ranked chunks without generation.
    # This is the same as semantic_retrieve() from Chapter 9 — LlamaIndex just
    # handles the chunking, embedding, and cosine similarity for us.
    retriever = index.as_retriever(similarity_top_k=4)
    nodes     = retriever.retrieve(QUERY)
    print(f"Query : {QUERY}")
    print(f"Top {len(nodes)} semantically similar chunks:\n")
    for node in nodes:
        fname = node.metadata.get("file_name", "?")
        print(f"  score={node.score:.3f}  {fname}")
        print(f"  {node.text[:200].strip()!r}")
        print()

---
## You're done.

You've built a fully functional local coding agent from scratch, one capability at a time.

| Ch | What you built | Key functions |
|----|---------------|---------------|
| 2  | LLM connection + status panel | `chat()`, `show_panel()`, `ping_ollama()` |
| 3  | Token budget awareness | `count_tokens()`, `scan_repo_costs()` |
| 4  | Project manifest | `load_manifest()`, `ask_with_manifest()` |
| 5  | File navigation without bulk loading | `glob_files()`, `jit_read()`, `budget_load()` |
| 6  | Relevance ranking by filename | `score_file()`, `rank_files()` |
| 7  | Content-based retrieval | `grep_repo()`, `grep_rank()` |
| 8  | Long-session survival | `eviction_candidates()`, `compact()`, `summarise_file()` |
| 9  | Meaning-based retrieval | `embed()`, `semantic_retrieve()` |
| 10 | Unified pipeline | `run()`, `pick_strategy()` |
| 11 | File modification with diff preview | `write_file()`, `make_diff()` |
| 12 | Autonomous task execution | `agent_loop()`, `plan_task()` |
| 13 | Self-verifying test generation | `test_loop()`, `generate_tests()` |
| 14 | Extending the agent | `fetch_url()`, three-step pattern |
| 15 | Reflection (ReAct loop) | `reflect()`, `agent_loop_reflect()`, `compact_thread()` |

### What to try next

- Point `REPO_ROOT` and `run()` at a real project you're working on
- Swap `OLLAMA_MODEL` for a larger model (`qwen4.5:14b`, `devstral-small-2`) and compare output quality
- Extend `plan_task()` to support a `refactor` action type
- Persist the embedding index to disk so it survives notebook restarts
- Add a `AGENTS.md` to your own repos


### Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `Connection refused` on first cell | Ollama isn't running | Run `ollama serve` in a terminal, then re-run the ping cell |
| `model not found` | Model name typo or not pulled | Run `ollama list` — copy the exact name shown |
| Token budget exceeded before Chapter 8 | Repo too large | Point `REPO_ROOT` at a smaller subfolder, or reduce `TOKEN_BUDGET` |
| Agent plans a READ step with a URL as the target | Model ignored planning rules | Re-run — the post-planning filter in `plan_task()` removes it automatically |
| `write_file` shows "no changes proposed" | Model returned an empty patch | Make the instruction more specific, or re-run |
| `REBUILD_INDEX = True` isn't rebuilding | Cell already ran with `False` | Set `REBUILD_INDEX = True` and re-run the `build_index` cell |
| `ping_ollama()` OK but `chat()` hangs | Wrong model name in `_ollama.model` | Verify `_ollama.model` matches an entry in `ollama list` |
| Tests generate but immediately fail | Model wrote wrong import paths | Re-run — `test_loop()` retries up to `max_retries` times |
| `agent_loop_reflect` never stops | Model keeps returning REVISE | Add a `max_revisions` guard or switch to a stronger model |
| Colab disconnects mid-run | Session timeout | Re-run from Chapter 1; Colab AI restores its own state |
